# WOWROBO SO-ARM101: RGB picking and scheduling experiments

This notebook downloads assets, constructs the workcell, runs an Isaac sample pick, detects objects with wrist RGB,
measures observation motion, physical pick times, and success rates, trains PPO/DQN, compares them with eight primary
baselines and handling-mode controls, validates policies in physics, runs tests, and saves results.

**Research question:** With the same RGB pick executor, how do scheduling and normal/careful mode selection
affect on-time completion, lateness, travel, and pick failures?

The workcell uses one follower arm, two input trays with four pockets each, an output location for each tray,
and one wrist-mounted RGB camera. Objects are red blocks of the same size. Order specifications provide
mass, handling risk, and deadlines. Camera policies receive RGB detections and order specifications.
Handling risk is modeled as an order attribute.

The notebook uses LeHome SO101 USD and URDF assets and the Isaac Sim Core API to construct the tray workcell
and pick executor.

**Runtime:** Both the notebook kernel and the physics worker use Python 3.12. The physics worker runs in an
Isaac Sim 6.0.0.1 environment. Two processes separate SimulationApp startup and shutdown from the notebook.
Run both on the same Linux or Windows machine with an RTX GPU. See the official requirements below for supported
GPUs and drivers. Training uses an event environment fitted to measurements; final validation and testing use
the PhysX+RGB executor.

`MODE="quick"` checks connectivity and the training path. Use `MODE="full"` for paper or assignment results.
Results are saved under `runs/vision_workcell/v2/{observation_profile}/{mode}/`.


## 1. Installation and runtime settings

Both the notebook kernel and the Isaac Sim worker use **Python 3.12**.
Install `isaacsim[all,extscache]==6.0.0.1` following the
[official 6.0 installation guide](https://docs.isaacsim.omniverse.nvidia.com/6.0.0/installation/install_python.html).
First install the CUDA version of PyTorch specified in that guide. Also install `opencv-python-headless==4.11.0.86`
and `imageio-ffmpeg==0.6.0` in the worker environment. Use the NumPy version required by Isaac Sim.

`ISAAC_COMMAND` is the path to the Python 3.12 executable in the Isaac Sim environment, or the distribution's
`python.sh`/`python.bat`. If Isaac Sim is installed in the notebook environment, set it to `[sys.executable]`.
The `%pip` cell below installs packages for the notebook kernel. Skip it if the same packages are already installed.

The camera uses the 6.0 `RtxCamera` and `CameraSensor` APIs. The workcell and joint control use the compatible
Core APIs included in 6.0. The asset download cell retrieves the SO101 USD and URDF files.


In [ ]:
%pip install "numpy>=1.26,<3" "gymnasium>=1.0,<2" "torch>=2.2,<3" "sb3-contrib==2.4.0" "stable-baselines3==2.4.1" "matplotlib>=3.8,<4" "pandas>=2.2,<3" "opencv-python-headless==4.11.0.86" "imageio-ffmpeg==0.6.0" "ortools>=9.11,<10" "scipy>=1.13,<2"


In [ ]:
import atexit
import copy
import random
import tempfile
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import secrets
import subprocess
import sys
import time
import traceback
import urllib.request
import xml.etree.ElementTree as ET
from collections import deque
from dataclasses import asdict, dataclass, field, replace
from multiprocessing.connection import Client, Listener
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from gymnasium import spaces
from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv
from torch import nn

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')

MODE = 'full'  # 'quick' uses a smaller budget for an execution check.
EXPERIMENT_VERSION = 2  # Bump when perception, execution, reward, or policy logic changes.
if MODE not in {'full', 'quick'}:
    raise ValueError("Set MODE to 'full' or 'quick'")
ISAAC_RUNTIME = {'package_version': '6.0.0.1', 'core_version': [6, 0, 0],
                 'python_version': [3, 12]}
if sys.version_info[:2] != tuple(ISAAC_RUNTIME['python_version']):
    raise RuntimeError('Use a Python 3.12 notebook kernel')
ISAAC_COMMAND = [os.environ.get('SOARM_ISAAC_PYTHON', '/opt/isaacsim/python.sh')]
HEADLESS = True

def canonical_hash(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()
    return hashlib.sha256(payload).hexdigest()

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def atomic_json(path, value):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    tmp.replace(path)


### 1.1. Full settings and quick overrides

Set `MODE` in the setup cell. The next cell defines the full experiment and shared settings;
the following cell applies every quick-mode override together. Run both in order, then run
the paths cell. Training, calibration, and evaluation use the resolved settings below.

After changing `MODE`, rerun these cells and the configuration cell before starting the experiment.


In [ ]:
# Full experiment defaults. Run this cell before the quick overrides.
SEEDS = list(range(10))
PPO_TIMESTEPS = DQN_TIMESTEPS = 300_000
VAL_EVERY = 25_000
N_VAL_SHIFTS = 20
N_TEST_SHIFTS = 100
CALIBRATION_REPEATS = 12
SHIFT_LENGTH = 480.0
CHECKPOINT_EVERY = 25_000
ROLLING_CP_SECONDS = 0.2
CLAIRVOYANT_CP_SECONDS = 5.0
DQN_TRAINING_OVERRIDES = {}
TEST_CONDITION_NAMES = (
    'familiar', 'unseen_p5', 'unseen_p6', 'busy', 'high_damage',
    'damage_out_of_range', 'appearance_dim', 'appearance_color',
)
REPORT_MODE_NOTES = ()

# Shared settings for both modes.
VAL_SEED, TEST_SEED, CALIBRATION_SEED = 20_000, 10_000, 30_000
RETRAIN = False
OBSERVATION_PROFILE = 'scheduling'  # Default: 79 values. 'vision': optional 95-value comparison.
RUN_TRAINING_SMOKE = True


In [ ]:
# All quick-mode changes are applied here, before paths and configuration hashes.
if MODE == 'quick':
    SEEDS = [0]
    PPO_TIMESTEPS = DQN_TIMESTEPS = 2_048
    VAL_EVERY = 1_024
    N_VAL_SHIFTS = 2
    N_TEST_SHIFTS = 2
    CALIBRATION_REPEATS = 4
    SHIFT_LENGTH = 30.0
    CHECKPOINT_EVERY = 512
    ROLLING_CP_SECONDS = 0.05
    CLAIRVOYANT_CP_SECONDS = 0.2
    DQN_TRAINING_OVERRIDES = dict(learning_starts=128, batch_size=32, buffer_size=4096)
    TEST_CONDITION_NAMES = ('familiar', 'damage_out_of_range', 'appearance_color')
    REPORT_MODE_NOTES = ('Quick mode uses a reduced budget for an execution check.',)


In [ ]:
RUN_DIR = Path('runs/vision_workcell') / f'v{EXPERIMENT_VERSION}' / OBSERVATION_PROFILE / MODE
RESULTS_DIR = RUN_DIR / 'results'
ASSET_DIR = RUN_DIR / 'assets'
WORKER_DIR = RUN_DIR / 'worker'
SHARED_MODULE_PATH = WORKER_DIR / 'vision_workcell_common.py'
WORKER_PATH = WORKER_DIR / 'isaac_worker.py'
for directory in (RUN_DIR, RESULTS_DIR, ASSET_DIR, WORKER_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print({'mode': MODE, 'experiment_version': EXPERIMENT_VERSION,
       'device': DEVICE, 'isaac_command': ISAAC_COMMAND,
       'seeds': SEEDS, 'test_shifts': N_TEST_SHIFTS})


## 2. Experiment specification and wrist camera

The camera attaches to the **fixed gripper body** after wrist_roll, following the
[SO101 wrist mounting guide](https://github.com/TheRobotStudio/SO-ARM100/blob/5f6d2b876a53a4872e405b991dd925556c9e38a4/Optional/SO101_Wrist_Cam_Hex-Nut_Mount_32x32_UVC_Module/README.md).
It does not attach to a moving gripper finger. The listed WOWROBO module specifications are
**2MP, 30 FPS, USB, and a 3m cable**. The exact capture mode, FOV, intrinsics, and optical-center position
require hardware measurements.

The initial `camera_local_xyz` value references the **bracket reference point** `wrist_camera_mount_joint` in the
[public SO101 camera URDF](https://github.com/TheRobotStudio/SO-ARM100/blob/5f6d2b876a53a4872e405b991dd925556c9e38a4/Simulation/SO101/so101_new_calib_camera.urdf).
Placing the sensor center there and aligning it with the gripper axis are simulation assumptions, not measured
extrinsics for the WOWROBO lens. The 640×480 resolution, lens/aperture settings, and zero distortion are also
model settings. The additional camera and bracket mass and geometry are not modeled.

World +Z points upward; units are m, s, kg, and rad. USD camera axes and OpenCV axes are converted explicitly.
For each frame, the camera pose is calculated from **measured joint angles → USD-based FK → gripper–camera transform**.
RGB pixel rays intersect the known block-top plane to recover positions. Objects of different heights require
a separate model.

On hardware, calibrate K, distortion, and the hand-eye transform. `calibrate_wrist_hand_eye` takes encoder FK
and solvePnP marker poses. It requires several poses with rotations around at least two nonparallel axes.
The current simulation uses the specified transform and checks agreement between FK and the sensor hierarchy,
plus an independent RGB position-error check. Regenerate measurements and checkpoints after changing calibration.
Careful mode only reduces loaded-motion speed to 1/1.5 of normal speed.

The hand-eye function returns an OpenCV camera→gripper transform. To use it as a USD camera→gripper transform,
postmultiply the output matrix by `USD_TO_CV`. Capture images at a stationary pose after motion and settling.


In [ ]:
PRODUCTS = {'P1': (0.03, 0.10), 'P2': (0.08, 0.60), 'P3': (0.18, 0.20),
            'P4': (0.05, 0.85), 'P5': (0.12, 0.45), 'P6': (0.23, 0.95)}
FAMILIAR = ('P1', 'P2', 'P3', 'P4')
EVAL_WEIGHTS = (0.015, 0.15, 0.6)
ACTION_NAMES = ('shoulder_pan', 'shoulder_lift', 'elbow_flex',
                'wrist_flex', 'wrist_roll', 'gripper')
HF_REVISION = 'bce78287bf9510449fbe4a045ebb6d53ad4f3ed0'
LEHOME_REVISION = '7613c74b4b85142d4d605f395b3156248feb7fbf'
USD_SHA256 = 'af8d2f281cff5c3bd3478fbce9341782979ce042872ca2a9a0638934a5bfa46f'

@dataclass(frozen=True)
class CellConfig:
    physics_dt: float = 1 / 120
    camera_fps: int = 30
    image_size: tuple = (640, 480)
    camera_mount: str = 'wrist'
    camera_local_xyz: tuple = (-0.015, 0.024, -0.0315)
    camera_local_rpy: tuple = (0.0, 0.0, 0.0)
    camera_distortion: tuple = (0.0, 0.0, 0.0, 0.0, 0.0)
    view_z: float = 0.120
    reobserve_z: float = 0.060
    focal_length: float = 0.024
    horizontal_aperture: float = 0.036
    table_z: float = 0.0
    block_size: float = 0.025
    hover_z: float = 0.085
    grasp_z: float = 0.030
    object_jitter: float = 0.004
    roi_radius: float = 0.021
    output_radius: float = 0.030
    gripper_open: float = 1.0
    gripper_closed: float = 0.1
    normal_speed: float = 1.5
    careful_slow: float = 1.5
    close_seconds: float = 0.35
    settle_seconds: float = 0.20
    motion_tolerance: float = 0.05
    kp: float = 17.8
    kd: float = 0.6
    effort_limit: float = 2.94
    contact_friction: float = 0.8
    inference_seconds: float = 0.020
    confidence_threshold: float = 0.35
    max_recaptures: int = 3
    feeder_seconds: float = 0.35
    clearance_seconds: float = 0.35
    appearance: str = 'nominal'

    def pockets(self):
        # Two 2x2 trays, all in the SO101 top-down reachable workspace.
        return np.array([[x, sign * y, self.table_z + self.block_size / 2]
                         for sign in (1, -1) for x in (0.078, 0.116)
                         for y in (0.140, 0.178)], dtype=float)

    def outputs(self):
        return np.array([[0.164, sign * 0.110, self.table_z + self.block_size / 2]
                         for sign in (1, -1)], dtype=float)

    def parks(self):
        return np.array([[0.16, sign * 0.045, self.hover_z] for sign in (1, -1)])

    def views(self):
        centers = self.pockets().reshape(2, 4, 3).mean(axis=1)
        centers[:, 2] = self.view_z
        return centers

@dataclass(frozen=True)
class EnvConfig:
    shift_length: float = 480.0
    arrival_rate: float = 0.14
    arrival_rate_range: tuple | None = (0.07, 0.21)
    initial_jobs: int = 3
    deadline_slack: tuple = (5.0, 50.0)
    num_trays: int = 2
    num_slots: int = 8
    careful_mode: bool = True
    max_attempts: int = 2
    on_time_bonus: float = 1.0
    unfinished_penalty: float = 1.0
    failure_penalty: float = 1.0
    scrap_penalty: float = 1.0
    proc_scale: float = 15.0
    weight_scale: float = 0.25
    deadline_scale: float = 60.0
    wait_scale: float = 120.0
    max_queue_obs: int = 64
    feature_low: tuple = (0.02, 0.0)
    feature_high: tuple = (0.20, 0.90)
    weight_low: tuple = (0.005, 0.05, 0.20)
    weight_high: tuple = (0.030, 0.30, 1.50)

CELL, ENV = CellConfig(), EnvConfig(shift_length=SHIFT_LENGTH)
if OBSERVATION_PROFILE not in {'vision', 'scheduling'}:
    raise ValueError("OBSERVATION_PROFILE must be 'vision' or 'scheduling'")
SLOT_FEATURES = ('present', 'processing_time', 'mass', 'handling_risk', 'deadline_slack',
                 'tray_A', 'tray_B', 'waiting_time', 'retried')
if OBSERVATION_PROFILE == 'vision':
    SLOT_FEATURES += ('rgb_confidence', 'processing_time_std')
GLOBAL_FEATURES = ('shift_remaining', 'arm_A', 'arm_B', 'backlog', 'alpha', 'beta', 'delta')
OBS_SCHEMA = {'version': 2, 'profile': OBSERVATION_PROFILE, 'slots': 8,
              'slot_dim': len(SLOT_FEATURES), 'global_dim': len(GLOBAL_FEATURES),
              'slot_features': SLOT_FEATURES, 'global_features': GLOBAL_FEATURES,
              'actions': 'mode * 8 + physical_pocket',
              'total_dim': 8*len(SLOT_FEATURES)+len(GLOBAL_FEATURES)}
print('Observation:', OBS_SCHEMA['total_dim'], '=', 8, '*', len(SLOT_FEATURES), '+', len(GLOBAL_FEATURES))
CAMERA_REFERENCE = {
    'repository': 'TheRobotStudio/SO-ARM100',
    'revision': '5f6d2b876a53a4872e405b991dd925556c9e38a4',
    'path': 'Simulation/SO101/so101_new_calib_camera.urdf',
    'mount_reference_joint': 'wrist_camera_mount_joint',
    'optical_extrinsics': 'simulation assumption; replace with measured hand-eye calibration',
}
SOURCE_SPEC = {'experiment_version': EXPERIMENT_VERSION,
               'lehome_revision': LEHOME_REVISION, 'hf_revision': HF_REVISION,
               'usd_sha256': USD_SHA256, 'cell': asdict(CELL), 'env': asdict(ENV),
               'obs': OBS_SCHEMA, 'mode': MODE, 'isaac_runtime': ISAAC_RUNTIME,
               'products': PRODUCTS, 'familiar': FAMILIAR, 'eval_weights': EVAL_WEIGHTS,
               'camera_reference': CAMERA_REFERENCE}


## 3. Assets and provenance

Download only the SO101 USD and URDF assets from the same LeHome release.
The URDF provides the link-chain and TCP reference. Joint origins and limits are read from the USD and used in FK/IK.
The two assets are not assumed to have identical joint calibration. PhysX computes contact, gravity, and object motion.
Verify the USD SHA-256 and save the URDF and settings hashes. The dedicated worker is generated from code in this notebook.


In [ ]:
def download_asset(name, expected_hash=None):
    path = ASSET_DIR / name
    if not path.exists():
        url = f'https://huggingface.co/datasets/lehome/lehome_release/resolve/{HF_REVISION}/robots/{name}'
        temporary = path.with_suffix(path.suffix + '.download')
        try:
            with urllib.request.urlopen(url, timeout=120) as response, temporary.open('wb') as target:
                while chunk := response.read(1024 * 1024):
                    target.write(chunk)
            temporary.replace(path)
        finally:
            temporary.unlink(missing_ok=True)
    digest = file_hash(path)
    if expected_hash and digest != expected_hash:
        raise RuntimeError(f'Asset hash mismatch: {name}. Remove this downloaded file and retry.')
    return path.resolve(), digest

USD_PATH, usd_hash = download_asset('so101_follower_good.usd', USD_SHA256)
URDF_PATH, urdf_hash = download_asset('so101_new_calib.urdf')
ASSET_MANIFEST = {'usd': usd_hash, 'urdf': urdf_hash, 'revision': HF_REVISION}
atomic_json(RUN_DIR / 'asset_manifest.json', ASSET_MANIFEST)
print(ASSET_MANIFEST)


## 4. Moving-camera perception and IK

Move to each input tray's observation pose and inspect its four pockets with RGB. The policy receives the latest
observations from sequential scans of the two trays; the pockets are not assumed to be visible simultaneously.
The order-to-pocket mapping provides the supply locations. The color detector converts pixels to coordinates
using the camera pose at capture time and the known block height. The camera executor does not receive depth,
simulator segmentation, or the target object's ground-truth pose.

After selecting a job, move above its pocket and capture an image. Capture another image at the approach pose
to refine xy before picking. Color and lighting shifts test this detector's limits. Apply any detector change
to all policies and repeat calibration.

The `%%writefile` cell below generates the shared perception and FK code. The next cell imports the module.


In [ ]:
%%writefile "$SHARED_MODULE_PATH"
import math
import xml.etree.ElementTree as ET
from dataclasses import dataclass
import numpy as np

def rotation(axis, angle):
    axis = np.asarray(axis, dtype=float)
    axis /= np.linalg.norm(axis)
    x, y, z = axis
    skew = np.array([[0, -z, y], [z, 0, -x], [-y, x, 0]])
    return np.eye(3) + math.sin(angle) * skew + (1 - math.cos(angle)) * skew @ skew

def origin_matrix(element):
    out = np.eye(4)
    if element is not None:
        out[:3, 3] = np.fromstring(element.get('xyz', '0 0 0'), sep=' ')
        roll, pitch, yaw = np.fromstring(element.get('rpy', '0 0 0'), sep=' ')
        out[:3, :3] = rotation([0, 0, 1], yaw) @ rotation([0, 1, 0], pitch) @ rotation([1, 0, 0], roll)
    return out

class URDFKinematics:
    def __init__(self, path, joint_overrides=None, base_transform=None):
        root = ET.parse(path).getroot()
        by_child = {j.find('child').get('link'): j for j in root.findall('joint')}
        chain, link = [], 'gripper_frame_link'
        while link in by_child:
            joint = by_child[link]
            chain.append(joint)
            link = joint.find('parent').get('link')
        self.chain = list(reversed(chain))
        self.tip_offset = origin_matrix(root.find("joint[@name='gripper_frame_joint']/origin"))
        self.overrides = joint_overrides or {}
        self.base_transform = np.eye(4) if base_transform is None else np.asarray(base_transform)
        self.names = [j.get('name') for j in self.chain if j.get('type') != 'fixed']
        self.low = np.array([float(j.find('limit').get('lower')) for j in self.chain if j.get('type') != 'fixed'])
        self.high = np.array([float(j.find('limit').get('upper')) for j in self.chain if j.get('type') != 'fixed'])
        for i, name in enumerate(self.names):
            if name in self.overrides:
                self.low[i], self.high[i] = self.overrides[name]['limits']
        if len(self.names) != 5:
            raise ValueError(f'SO101 requires five arm joints; found {self.names}')

    def fk(self, q):
        transform, i = self.base_transform.copy(), 0
        for joint in self.chain:
            override = self.overrides.get(joint.get('name'))
            transform = transform @ (np.asarray(override['origin']) if override else origin_matrix(joint.find('origin')))
            if joint.get('type') != 'fixed':
                step = np.eye(4)
                axis = override['axis'] if override else np.fromstring(joint.find('axis').get('xyz'), sep=' ')
                step[:3, :3] = rotation(axis, q[i])
                transform = transform @ step
                if override:
                    transform = transform @ np.asarray(override['child_inverse'])
                i += 1
        return transform

    def camera_pose(self, q, cell):
        # Encoder FK gives the rigid wrist pose; moving-jaw position is not part of this chain.
        wrist = self.fk(q) @ np.linalg.inv(self.tip_offset)
        return wrist @ camera_mount_transform(cell) @ USD_TO_CV

    def solve(self, xyz, seed=None):
        xyz = np.asarray(xyz, dtype=float)
        def error(q):
            transform = self.fk(q)
            return np.r_[xyz - transform[:3, 3], 0.12 * (np.array([0., 0., -1.]) - transform[:3, 2])]
        seeds = [np.asarray(seed[:5]).copy()] if seed is not None else []
        pan = -math.atan2(xyz[1], xyz[0])
        seeds += [np.array([pan, lift, elbow, wrist, 0.])
                  for lift in (-0.6, 0.4) for elbow in (-0.4, 0.9) for wrist in (0.4, 1.2)]
        best, best_norm = None, np.inf
        for q in seeds:
            q = np.clip(q, self.low, self.high)
            for _ in range(160):
                err = error(q)
                if np.linalg.norm(err) < 2e-4:
                    break
                epsilon = 1e-5
                jac = np.column_stack([(error(q + epsilon * np.eye(5)[k]) - err) / epsilon for k in range(5)])
                dq = -jac.T @ np.linalg.solve(jac @ jac.T + 1e-5 * np.eye(6), err)
                q = np.clip(q + np.clip(dq, -0.15, 0.15), self.low, self.high)
            norm = np.linalg.norm(error(q))
            if norm < best_norm:
                best, best_norm = q.copy(), norm
            if norm < 2e-4:
                break
        transform = self.fk(best)
        if np.linalg.norm(transform[:3, 3] - xyz) > 0.001 or transform[2, 2] > -math.cos(math.radians(3)):
            raise ValueError(f'Unreachable top-down target: {xyz.tolist()} (error={best_norm:.4f})')
        return best


# Camera USD: +X right, +Y up, -Z forward. OpenCV: +X right, +Y down, +Z forward.
USD_TO_CV = np.diag([1., -1., -1., 1.])

def camera_mount_transform(cell):
    transform = np.eye(4)
    roll, pitch, yaw = cell['camera_local_rpy']
    transform[:3, :3] = rotation([0, 0, 1], yaw) @ rotation([0, 1, 0], pitch) @ rotation([1, 0, 0], roll)
    transform[:3, 3] = cell['camera_local_xyz']
    return transform

def camera_mount_quaternion(cell):
    roll, pitch, yaw = np.asarray(cell['camera_local_rpy']) / 2
    cr, cp, cy = np.cos([roll, pitch, yaw])
    sr, sp, sy = np.sin([roll, pitch, yaw])
    return np.array([cr*cp*cy+sr*sp*sy, sr*cp*cy-cr*sp*sy,
                     cr*sp*cy+sr*cp*sy, cr*cp*sy-sr*sp*cy])

def scan_order(occupied, arm):
    return [tray for tray in (1-int(arm), int(arm)) if np.any(np.asarray(occupied)[tray*4:tray*4+4])]

def pixel_to_plane(uv, k, camera_to_world, plane_z, distortion):
    import cv2
    normalized = cv2.undistortPoints(np.asarray(uv, dtype=float).reshape(1, 1, 2), k, distortion)[0, 0]
    origin = camera_to_world[:3, 3]
    ray = camera_to_world[:3, :3] @ np.r_[normalized, 1.]
    if abs(ray[2]) < 1e-9:
        return None
    distance = (plane_z-origin[2]) / ray[2]
    if distance <= 0:
        return None
    return (origin + distance*ray)[:2]

def world_to_pixel(xyz, k, camera_to_world, distortion):
    import cv2
    camera_point = np.linalg.inv(camera_to_world) @ np.r_[xyz, 1.]
    if camera_point[2] <= 0:
        return None
    pixel, _ = cv2.projectPoints(camera_point[:3].reshape(1, 3), np.zeros(3), np.zeros(3), k, distortion)
    return pixel[0, 0]

def calibrate_wrist_hand_eye(gripper_to_base, target_to_camera):
    # Arguments are transforms gripper->base and target->OpenCV camera respectively.
    # Real hardware supplies diverse encoder poses and solvePnP board poses.
    import cv2
    hands = np.asarray(gripper_to_base, dtype=float)
    targets = np.asarray(target_to_camera, dtype=float)
    if hands.shape != targets.shape or hands.ndim != 3 or hands.shape[1:] != (4, 4) or len(hands) < 3:
        raise ValueError('Hand-eye calibration needs at least three paired 4x4 poses')
    axes = [cv2.Rodrigues(h[:3, :3] @ hands[0, :3, :3].T)[0].ravel() for h in hands[1:]]
    if np.linalg.matrix_rank(np.asarray(axes), tol=1e-5) < 2:
        raise ValueError('Use motions with at least two non-parallel rotation axes')
    r, t = cv2.calibrateHandEye(list(hands[:, :3, :3]), list(hands[:, :3, 3]),
                              list(targets[:, :3, :3]), list(targets[:, :3, 3]),
                              method=cv2.CALIB_HAND_EYE_PARK)
    result = np.eye(4)
    result[:3, :3], result[:3, 3] = r, t.ravel()
    if not np.isfinite(result).all():
        raise ValueError('Hand-eye calibration produced invalid values')
    return result  # OpenCV camera -> rigid gripper body

class RGBDetector:
    def __init__(self, cell, intrinsics):
        self.cell = cell
        self.k = np.asarray(intrinsics, dtype=float)
        self.distortion = np.asarray(cell['camera_distortion'], dtype=float)
        if self.k.shape != (3, 3) or not np.isfinite(self.k).all() or min(self.k[0, 0], self.k[1, 1]) <= 0:
            raise ValueError('Invalid RGB camera intrinsics')

    def detect(self, rgb, center, radius, camera_to_world):
        import cv2
        image = np.asarray(rgb, dtype=np.uint8)
        r, g, b = [image[..., k].astype(float) for k in range(3)]
        mask = ((r > 55) & (r > 1.6*g) & (r > 1.6*b)).astype(np.uint8)
        n, _, stats, centers = cv2.connectedComponentsWithStats(mask)
        choices = []
        for i in range(1, n):
            area = stats[i, cv2.CC_STAT_AREA]
            if area < 8:
                continue
            # Intersect the current camera ray with the known block-top plane.
            # This uses RGB, encoder FK, and calibration; no depth or object pose is read.
            xy = pixel_to_plane(centers[i], self.k, camera_to_world,
                                self.cell['table_z']+self.cell['block_size'], self.distortion)
            if xy is None:
                continue
            distance = np.linalg.norm(xy-np.asarray(center)[:2])
            if distance <= radius:
                confidence = min(1., area/45.) * max(0., 1.-0.4*distance/radius)
                choices.append({'xy': xy.tolist(), 'confidence': float(confidence), 'area': int(area)})
        return max(choices, key=lambda x: x['confidence']) if choices else {'xy': None, 'confidence': 0., 'area': 0}

def look_at_quaternion(eye, target):
    eye, target = np.asarray(eye, dtype=float), np.asarray(target, dtype=float)
    if eye.shape != (3,) or target.shape != (3,) or not np.isfinite([eye, target]).all():
        raise ValueError('Camera eye and target must be finite xyz vectors')
    backward = eye - target
    if np.linalg.norm(backward) < 1e-8:
        raise ValueError('Camera eye and target must differ')
    backward /= np.linalg.norm(backward)
    up = np.array([0., 0., 1.])
    if abs(np.dot(up, backward)) > .99:
        up = np.array([0., 1., 0.])
    right = np.cross(up, backward)
    right /= np.linalg.norm(right)
    up = np.cross(backward, right)
    matrix = np.column_stack([right, up, backward])
    roll = math.atan2(matrix[2, 1], matrix[2, 2])
    pitch = math.atan2(-matrix[2, 0], np.hypot(matrix[0, 0], matrix[1, 0]))
    yaw = math.atan2(matrix[1, 0], matrix[0, 0])
    return camera_mount_quaternion({'camera_local_rpy': (roll, pitch, yaw)})


class WorkcellVideoWriter:
    def __init__(self, path, image_size, fps, label):
        from pathlib import Path
        import imageio_ffmpeg
        self.path = Path(path)
        if self.path.suffix.lower() != '.mp4':
            raise ValueError('Video output must be an MP4 path')
        self.width, self.height = map(int, image_size)
        self.fps = float(fps)
        if (self.width < 2 or self.height < 2 or self.width % 2 or self.height % 2
                or not np.isfinite(self.fps) or self.fps <= 0):
            raise ValueError('Use positive even image dimensions and a positive FPS')
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.partial_path = self.path.with_name(self.path.stem + '.partial.mp4')
        self.partial_path.unlink(missing_ok=True)
        self.label = str(label)
        self.frames, self.first_time, self.last_time = 0, None, None
        self._encoder, self._closed = None, False
        self._ffmpeg = imageio_ffmpeg

    def write(self, overview, wrist, timestamp, action):
        import cv2
        if self._closed:
            raise RuntimeError('Video writer is closed')
        shape = (self.height, self.width, 3)
        overview, wrist = np.asarray(overview), np.asarray(wrist)
        if (overview.shape != shape or wrist.shape != shape
                or overview.dtype != np.uint8 or wrist.dtype != np.uint8):
            raise ValueError('Both video views must be uint8 RGB images at the configured size')
        timestamp = float(timestamp)
        if not np.isfinite(timestamp) or timestamp < 0 or (self.frames and timestamp <= self.last_time):
            raise ValueError('Video timestamps must increase')
        frame = np.zeros((self.height + 48, 2 * self.width, 3), dtype=np.uint8)
        frame[48:, :self.width] = overview
        frame[48:, self.width:] = wrist
        cv2.putText(frame, f'{self.label} | t={timestamp:.2f}s | {action}', (10, 18),
                    cv2.FONT_HERSHEY_SIMPLEX, .45, (255, 255, 255), 1, cv2.LINE_AA)
        cv2.putText(frame, 'Workcell overview', (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, .45, (255, 255, 255), 1, cv2.LINE_AA)
        cv2.putText(frame, 'Wrist RGB', (self.width + 10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, .45, (255, 255, 255), 1, cv2.LINE_AA)
        if self._encoder is None:
            self._encoder = self._ffmpeg.write_frames(str(self.partial_path),
                size=(2 * self.width, self.height + 48), fps=self.fps,
                codec='libx264', pix_fmt_in='rgb24', pix_fmt_out='yuv420p',
                macro_block_size=2, ffmpeg_log_level='error', ffmpeg_timeout=30,
                output_params=['-movflags', '+faststart'])
            self._encoder.send(None)
        self._encoder.send(frame)
        if not self.frames:
            self.first_time = timestamp
        self.frames += 1
        self.last_time = timestamp

    def close(self):
        if self._closed:
            raise RuntimeError('Video writer is closed')
        self._closed = True
        try:
            if not self.frames:
                raise RuntimeError('No RGB frames were recorded')
            self._encoder.close()
            self._encoder = None
            frames, seconds = self._ffmpeg.count_frames_and_secs(str(self.partial_path))
            if frames != self.frames:
                raise RuntimeError(f'Video is incomplete: {frames}/{self.frames} frames')
            self.partial_path.replace(self.path)
            return {'path': str(self.path), 'frames': self.frames, 'fps': self.fps,
                    'duration_seconds': float(seconds), 'frame_size': [2 * self.width, self.height + 48],
                    'first_sim_time': self.first_time, 'last_sim_time': self.last_time,
                    'codec': 'h264', 'views': ['workcell_overview', 'wrist_rgb']}
        except BaseException:
            self.abort()
            raise

    def abort(self):
        self._closed = True
        encoder, self._encoder = self._encoder, None
        try:
            if encoder is not None:
                encoder.close()
        finally:
            self.partial_path.unlink(missing_ok=True)


In [ ]:
import importlib.util

# Load the generated module for this mode, including edits made in this session.
Path(importlib.util.cache_from_source(str(SHARED_MODULE_PATH))).unlink(missing_ok=True)
_common_spec = importlib.util.spec_from_file_location('vision_workcell_common', SHARED_MODULE_PATH)
_common_module = importlib.util.module_from_spec(_common_spec)
sys.modules[_common_spec.name] = _common_module
_common_spec.loader.exec_module(_common_module)

from vision_workcell_common import (
    RGBDetector,
    URDFKinematics,
    USD_TO_CV,
    WorkcellVideoWriter,
    calibrate_wrist_hand_eye,
    camera_mount_quaternion,
    camera_mount_transform,
    look_at_quaternion,
    origin_matrix,
    pixel_to_plane,
    rotation,
    scan_order,
    world_to_pixel,
)


## 5. PhysX pick executor

The worker constructs the robot USD, rigid blocks, trays, and wrist RGB camera. At startup, it checks that FK
with the USD joint transforms matches actual link coordinates and that IK can reach every pocket and output location.
Scene reset and the feeder place the initial scene and newly supplied orders. During picking, objects are not
attached to the gripper or moved to ground-truth positions; gripper contact and friction move them.

Joint tracking outside the allowed tolerance or missing RGB is recorded as a failure.
Normal and careful modes share the same path and gripper settings. The camera executor localizes with RGB;
the oracle control uses positions from the evaluator. Success requires both RGB detection in the output area
and evaluator confirmation that the target object reached that area. Evaluator coordinates are not returned
as policy observations.

The clock uses physics time, including RGB capture intervals, configured inference latency, and supply/removal time.
Rendering and IPC wall-clock time do not enter the reward. Operations that exceed 480 seconds are stopped
and counted as unfinished.

Observation scans visit the other tray before the current tray, scan the required trays, and return to the starting
station's park pose. Observation motion, settling, and capture time count toward the shift clock. Station switches
during observation also incur the β cost. Output verification uses an observation pose above the output area.
Camera and oracle executors follow the same capture and motion procedure.

The next `%%writefile` cell generates `isaac_worker.py`. The worker and shared module are saved in
`RUN_DIR / "worker"`; the worker imports that same shared module.


In [ ]:
%%writefile "$WORKER_PATH"
import argparse
import importlib.metadata
import json
import math
import platform
import sys
import traceback
import xml.etree.ElementTree as ET
from multiprocessing.connection import Client
import numpy as np

parser = argparse.ArgumentParser()
parser.add_argument('--config', required=True)
parser.add_argument('--port', type=int, required=True)
parser.add_argument('--auth', required=True)
args = parser.parse_args()
spec = json.load(open(args.config))
if sys.version_info[:2] != tuple(spec['runtime']['python_version']):
    raise RuntimeError('The Isaac Sim 6.0 worker requires a Python 3.12 environment')
from isaacsim import SimulationApp
application = SimulationApp({'headless': spec['headless'], 'width': 640, 'height': 480})
from isaacsim.core.api import World
from isaacsim.core.api.objects import DynamicCuboid, FixedCuboid
from isaacsim.core.api.materials import PhysicsMaterial
from isaacsim.core.prims import SingleArticulation
from isaacsim.core.utils.stage import add_reference_to_stage
from isaacsim.core.utils.types import ArticulationAction
from isaacsim.sensors.experimental.rtx import CameraSensor, RtxCamera
from isaacsim.core.version import get_version
from pxr import Usd, UsdGeom, UsdLux, UsdPhysics, PhysxSchema, Gf
from vision_workcell_common import (
    RGBDetector,
    URDFKinematics,
    USD_TO_CV,
    WorkcellVideoWriter,
    camera_mount_quaternion,
    camera_mount_transform,
    look_at_quaternion,
    origin_matrix,
    scan_order,
)

class WorkcellRGBCamera:
    def __init__(self, config, parent_path, name='WristRGBCamera', position=None, orientation=None):
        self.width, self.height = config['image_size']
        self.path = parent_path + '/' + name
        self.authoring = RtxCamera(self.path, tick_rate=config['camera_fps'],
            translations=np.asarray([config['camera_local_xyz'] if position is None else position]),
            orientations=np.asarray([camera_mount_quaternion(config) if orientation is None else orientation]))
        self.authoring.camera.set_focal_lengths(config['focal_length'])
        self.authoring.camera.set_apertures(horizontal_apertures=config['horizontal_aperture'])
        self.sensor = CameraSensor(self.authoring, resolution=(self.height, self.width), annotators=['rgb'])

    def get_rgb(self):
        data, _ = self.sensor.get_data('rgb')
        if data is None:
            raise RuntimeError('No RTX RGB frame available')
        rgb = data.numpy()  # Warp copies CUDA data to a NumPy host array.
        if rgb.shape != (self.height, self.width, 3):
            raise RuntimeError(f'Unexpected RTX RGB frame: {rgb.shape}')
        return np.ascontiguousarray(rgb)

    def get_intrinsics_matrix(self):
        # Read authored lens values after CameraSensor has enforced square pixels.
        camera = self.authoring.camera
        focal = float(camera.get_focal_lengths().numpy().reshape(-1)[0])
        horizontal, vertical = camera.get_apertures()
        horizontal = float(horizontal.numpy().reshape(-1)[0])
        vertical = float(vertical.numpy().reshape(-1)[0])
        return np.array([[self.width * focal / horizontal, 0., self.width / 2],
                         [0., self.height * focal / vertical, self.height / 2],
                         [0., 0., 1.]])

def usd_joint_overrides(stage, joint_names):
    def local_transform(position, quaternion):
        out = np.asarray(Gf.Matrix4d().SetRotate(Gf.Rotation(quaternion))).T.copy()
        out[:3, 3] = np.asarray(position)
        return out
    result = {}
    for prim in stage.Traverse():
        if prim.IsA(UsdPhysics.RevoluteJoint) and prim.GetName() in joint_names:
            joint = UsdPhysics.RevoluteJoint(prim)
            axis = {'X': [1., 0., 0.], 'Y': [0., 1., 0.], 'Z': [0., 0., 1.]}[str(joint.GetAxisAttr().Get())]
            result[prim.GetName()] = {
                'origin': local_transform(joint.GetLocalPos0Attr().Get(), joint.GetLocalRot0Attr().Get()).tolist(),
                'child_inverse': np.linalg.inv(local_transform(joint.GetLocalPos1Attr().Get(), joint.GetLocalRot1Attr().Get())).tolist(),
                'axis': axis,
                'limits': np.radians([joint.GetLowerLimitAttr().Get(), joint.GetUpperLimitAttr().Get()]).tolist()}
    if set(result) != set(joint_names):
        raise RuntimeError(f'Unexpected USD joint names: {set(result)}')
    return result

class BudgetExpired(Exception):
    pass

class DetectionFailed(Exception):
    pass

class Workcell:
    def __init__(self):
        self.c = spec['cell']
        if self.c['camera_mount'] != 'wrist':
            raise ValueError('This experiment requires a wrist-mounted camera')
        self.dt = self.c['physics_dt']
        self.world = World(stage_units_in_meters=1., physics_dt=self.dt, rendering_dt=1 / self.c['camera_fps'], backend='numpy')
        self.world.scene.add_default_ground_plane()
        self.stage = self.world.stage
        prims, base_world = self._load_robot()
        self.pockets = np.asarray(spec['pockets'])
        self.outputs = np.asarray(spec['outputs'])
        self.parks = np.asarray(spec['parks'])
        self.views = np.asarray(spec['views'])
        self._create_pockets_and_trays()
        light = UsdLux.DomeLight.Define(self.stage, '/World/Light')
        light.CreateIntensityAttr().Set(700.)
        self.light = light
        self.camera = WorkcellRGBCamera(self.c, str(self.tip_prim.GetPath()))
        self._configure_articulation(prims)
        self._recording, self._replay_camera, self._recording_error = None, None, None
        self._recording_action = 'Initial observation'
        self.occupied = np.zeros(8, dtype=bool)
        self.sim_steps, self.limit_step, self.arm = 0, None, 0
        self.detector = RGBDetector(self.c, self.camera.get_intrinsics_matrix())
        self.reset(0, 0, 'nominal')
        self._validate_kinematics()
        self.reset(0, 0, 'nominal')
        self.calibration = {'K': self.detector.k.tolist(), 'distortion': self.detector.distortion.tolist(),
            'resolution': list(self.c['image_size']), 'mount': 'wrist',
            'parent_link': str(self.tip_prim.GetPath()),
            'T_gripper_camera_usd': camera_mount_transform(self.c).tolist(),
            'extrinsics_source': 'authored simulation transform, checked against encoder FK',
            'joint_overrides': self.joint_overrides, 'base_transform': np.asarray(base_world).T.tolist()}
        self._validate_wrist_views()
        self.reset(0, 0, 'nominal')

    def _load_robot(self):
        # Rotate the USD's native forward direction (-Y) into this workcell's +X.
        mount = UsdGeom.Xform.Define(self.stage, '/World/Mount')
        mount.AddRotateZOp().Set(90.)
        add_reference_to_stage(spec['usd'], '/World/Mount/SO101')
        root = self.stage.GetPrimAtPath('/World/Mount/SO101')
        prims = list(Usd.PrimRange(root))
        articulation_roots = [p for p in prims if p.HasAPI(UsdPhysics.ArticulationRootAPI)]
        if len(articulation_roots) != 1:
            raise RuntimeError(f'Expected one articulation root, found {len(articulation_roots)}')
        self.robot = self.world.scene.add(SingleArticulation(str(articulation_roots[0].GetPath()), name='so101'))
        def link(name):
            matches = [p for p in prims if p.GetName() == name]
            if len(matches) != 1:
                raise RuntimeError(f'Missing or ambiguous USD link: {name}')
            return matches[0]
        self.base_prim, self.tip_prim = link('base'), link('gripper')
        # This USD 'gripper' body is the wrist_roll child, not the moving_jaw finger.
        roll_joint = next(UsdPhysics.RevoluteJoint(p) for p in prims
                          if p.IsA(UsdPhysics.RevoluteJoint) and p.GetName() == 'wrist_roll')
        if roll_joint.GetBody1Rel().GetTargets() != [self.tip_prim.GetPath()]:
            raise RuntimeError('Camera parent must be the rigid wrist_roll child')
        base_world = UsdGeom.XformCache().GetLocalToWorldTransform(self.base_prim)
        # Fix the root to the world and enable gravity for every robot body.
        if not any(p.IsA(UsdPhysics.FixedJoint) and not p.GetRelationship('physics:body0').GetTargets()
                   for p in prims):
            fixed = UsdPhysics.FixedJoint.Define(self.stage, '/World/RobotMount')
            fixed.CreateBody1Rel().SetTargets([self.base_prim.GetPath()])
            fixed.CreateLocalPos0Attr().Set(Gf.Vec3f(base_world.ExtractTranslation()))
            fixed.CreateLocalRot0Attr().Set(Gf.Quatf(base_world.ExtractRotationQuat()))
        for prim in prims:
            if prim.HasAPI(UsdPhysics.RigidBodyAPI):
                PhysxSchema.PhysxRigidBodyAPI.Apply(prim).CreateDisableGravityAttr().Set(False)
        self.joint_overrides = usd_joint_overrides(self.stage, spec['joint_names'])
        self.kin = URDFKinematics(spec['urdf'], self.joint_overrides, np.asarray(base_world).T)
        self.tip_offset = origin_matrix(ET.parse(spec['urdf']).getroot().find("joint[@name='gripper_frame_joint']/origin"))
        return prims, base_world

    def _create_pockets_and_trays(self):
        self.objects = []
        material = PhysicsMaterial('/World/ContactMaterial', static_friction=self.c['contact_friction'],
                                   dynamic_friction=self.c['contact_friction'], restitution=0.)
        for slot, xyz in enumerate(self.pockets):
            self.world.scene.add(FixedCuboid(f'/World/Pocket{slot}', name=f'pocket{slot}',
                position=xyz - np.array([0., 0., self.c['block_size']/2 + .002]),
                scale=np.array([.034, .034, .004]), color=np.array([.25, .25, .25]), physics_material=material))
            obj = self.world.scene.add(DynamicCuboid(f'/World/Block{slot}', name=f'block{slot}',
                position=np.array([5. + slot, 0., 1.]), scale=np.full(3, self.c['block_size']),
                mass=.08, color=np.array([.85, .035, .035]), physics_material=material))
            self.objects.append(obj)
        for tray, xyz in enumerate(self.outputs):
            self.world.scene.add(FixedCuboid(f'/World/Output{tray}', name=f'output{tray}',
                position=xyz - np.array([0., 0., self.c['block_size']/2 + .002]),
                scale=np.array([.065, .065, .004]), color=np.array([.30, .30, .30]), physics_material=material))

    def _configure_articulation(self, prims):
        self.world.reset()
        self.robot.initialize()
        self.indices = np.array([self.robot.dof_names.index(n) for n in spec['joint_names']], dtype=int)
        self.controller = self.robot.get_articulation_controller()
        self.controller.set_gains(kps=np.full(self.robot.num_dof, self.c['kp']), kds=np.full(self.robot.num_dof, self.c['kd']))
        # USD stores angular drive units in degrees, core commands are radians.
        for prim in prims:
            if prim.IsA(UsdPhysics.RevoluteJoint):
                drive = UsdPhysics.DriveAPI.Get(prim, 'angular')
                if drive:
                    drive.CreateMaxForceAttr().Set(self.c['effort_limit'])

    def _validate_kinematics(self):
        # Kinematic validity is a gate, not a silent coordinate correction.
        cache = UsdGeom.XformCache()
        for target in (self.outputs[0], self.outputs[1], self.pockets[0]):
            q = self.kin.solve([*target[:2], self.c['hover_z']])
            self.robot.set_joint_positions(np.r_[q, self.c['gripper_open']], joint_indices=self.indices)
            self.apply(np.r_[q, self.c['gripper_open']])
            self.advance(.10)
            cache.Clear()
            transform = np.asarray(cache.GetLocalToWorldTransform(self.tip_prim)).T @ self.tip_offset
            measured = self.robot.get_joint_positions(joint_indices=self.indices)
            expected = self.kin.fk(measured[:5])
            if np.linalg.norm(transform[:3, 3] - expected[:3, 3]) > .003:
                raise RuntimeError('FK and measured USD frames disagree by more than 3mm; do not run experiments')
            actual_camera = np.asarray(cache.GetLocalToWorldTransform(
                self.stage.GetPrimAtPath(self.camera.path))).T @ USD_TO_CV
            predicted_camera = self.kin.camera_pose(measured[:5], self.c)
            if (np.linalg.norm(actual_camera[:3, 3]-predicted_camera[:3, 3]) > .003
                    or np.linalg.norm(actual_camera[:3, :3]-predicted_camera[:3, :3]) > .02):
                raise RuntimeError('Wrist camera hierarchy and encoder FK disagree')
        for xyz in np.vstack([self.pockets, self.outputs, self.parks]):
            for z in (self.c['grasp_z'], self.c['hover_z']):
                self.kin.solve([*xyz[:2], z])
        for view in self.views:
            self.kin.solve(view)
        for xyz in self.pockets:
            self.kin.solve([*xyz[:2], self.c['reobserve_z']])
        for xyz in self.outputs:
            self.kin.solve([*xyz[:2], self.c['view_z']])

    def _validate_wrist_views(self):
        errors = []
        for slot in range(8):
            self.present(slot, .08, slot, jitter=False)
            for target in (self.views[slot//4], [*self.pockets[slot, :2], self.c['reobserve_z']]):
                self.move(target, self.c['gripper_open'], self.c['normal_speed'])
                detection = self.sense(self.pockets[slot], self.c['roi_radius'])
                if detection['xy'] is None:
                    raise RuntimeError(f'Pocket {slot} is not visible from wrist view {target}')
                errors.append(float(np.linalg.norm(np.asarray(detection['xy'])-self.pockets[slot, :2])))
            self.remove(slot)
        self.calibration['wrist_view_errors_m'] = errors
        if max(errors) > .004:
            raise RuntimeError(f'Moving RGB localization error exceeds 4mm: {errors}')

    def apply(self, q):
        self.robot.apply_action(ArticulationAction(joint_positions=np.asarray(q), joint_indices=self.indices))

    def advance(self, seconds):
        for _ in range(int(math.ceil(max(0., seconds) / self.dt - 1e-9))):
            if self.limit_step is not None and self.sim_steps >= self.limit_step:
                raise BudgetExpired()
            render = self.sim_steps % round(1 / self.c['camera_fps'] / self.dt) == 0
            self.world.step(render=render)
            self.sim_steps += 1
            if render and self._recording is not None and self.sim_steps >= self._recording_next_step:
                try:
                    self._recording.write(self._replay_camera.get_rgb(), self.camera.get_rgb(),
                        self.sim_steps*self.dt, self._recording_action)
                    self._recording_next_step = self.sim_steps + self._recording_stride
                except Exception as error:
                    self._recording_error = str(error)
                    recorder, self._recording = self._recording, None
                    try:
                        recorder.abort()
                    except Exception as cleanup_error:
                        self._recording_error += f'; encoder cleanup: {cleanup_error}'

    def start_recording(self, path, fps, label, eye, target):
        if self._recording is not None:
            raise RuntimeError('A video is already being recorded')
        fps = float(fps)
        if not math.isfinite(fps) or fps <= 0 or fps > self.c['camera_fps']:
            raise ValueError('Recording FPS must be positive and no higher than the wrist camera FPS')
        stride = round(1 / fps / self.dt)
        render_stride = round(1 / self.c['camera_fps'] / self.dt)
        if not math.isclose(stride * self.dt * fps, 1., abs_tol=1e-7) or stride % render_stride:
            raise ValueError('Recording FPS must align with physics and camera render steps')
        orientation = look_at_quaternion(eye, target)
        if self._replay_camera is None:
            view_config = self.c | {'focal_length': .018}
            self._replay_camera = WorkcellRGBCamera(view_config, '/World', name='ReplayCamera',
                position=eye, orientation=orientation)
        else:
            self._replay_camera.authoring.set_local_poses(
                translations=np.asarray([eye]), orientations=np.asarray([orientation]))
        # Warm the new render product without advancing physics time.
        for _ in range(3):
            self.world.render()
        self._recording = WorkcellVideoWriter(path, self.c['image_size'], fps, label)
        self._recording_stride, self._recording_next_step = stride, self.sim_steps
        self._recording_error, self._recording_action = None, 'Initial observation'
        return {'path': str(self._recording.path), 'fps': fps}


    def stop_recording(self):
        if self._recording_error is not None:
            raise RuntimeError(f'Video recording failed: {self._recording_error}')
        recorder, self._recording = self._recording, None
        if recorder is None:
            raise RuntimeError('No active video recording')
        return recorder.close()


    def abort_recording(self):
        recorder, self._recording = self._recording, None
        self._recording_error = None
        if recorder is not None:
            recorder.abort()
        return {'aborted': recorder is not None}

    def capture(self):
        self.advance(1 / self.c['camera_fps'] + self.c['inference_seconds'])
        rgb = np.asarray(self.camera.get_rgb())
        if rgb.shape != (self.c['image_size'][1], self.c['image_size'][0], 3):
            raise RuntimeError(f'No RGB frame: {rgb.shape}')
        return rgb

    def camera_pose(self):
        measured = self.robot.get_joint_positions(joint_indices=self.indices)
        return self.kin.camera_pose(measured[:5], self.c)

    def sense(self, center, radius):
        rgb = self.capture()
        return self.detector.detect(rgb, center, radius, self.camera_pose())

    def reset(self, seed, arm, appearance):
        if self._recording is not None:
            raise RuntimeError('Stop or abort the video before resetting the scene')
        self.limit_step, self.arm = None, int(arm)
        self.scene_seed, self.appearance = int(seed), appearance
        self.light.GetIntensityAttr().Set(400. if appearance == 'dim' else 700.)
        for i in range(8):
            self.remove(i)
        q = self.kin.solve(self.parks[self.arm])
        self.robot.set_joint_positions(np.r_[q, self.c['gripper_open']], joint_indices=self.indices)
        self.robot.set_joint_velocities(np.zeros(6), joint_indices=self.indices)
        self.apply(np.r_[q, self.c['gripper_open']])
        self.advance(.3)
        self.sim_steps = 0
        return {'elapsed': 0.}

    def present(self, slot, weight, seed, jitter=True):
        obj = self.objects[slot]
        rng = np.random.default_rng(seed)
        xyz = self.pockets[slot].copy()
        if jitter:
            xyz[:2] += rng.uniform(-self.c['object_jitter'], self.c['object_jitter'], 2)
        yaw = float(rng.uniform(-.2, .2))
        obj.set_visibility(True)
        obj.set_world_pose(position=xyz, orientation=np.array([math.cos(yaw/2), 0., 0., math.sin(yaw/2)]))
        obj.set_linear_velocity(np.zeros(3))
        obj.set_angular_velocity(np.zeros(3))
        obj.set_mass(float(weight))
        # Appearance shifts change the scene material, not the detector's input labels.
        material = obj.get_applied_visual_material()
        if material:
            material.set_color(np.array([.40, .16, .12]) if self.appearance == 'color' else np.array([.85, .035, .035]))
        self.occupied[slot] = True
        return {'elapsed': 0.}

    def remove(self, slot):
        obj = self.objects[slot]
        # An external conveyor/recovery removes a finished/scrapped block. Keep the actor alive.
        obj.set_world_pose(position=np.array([5. + slot, 0., .1]))
        obj.set_linear_velocity(np.zeros(3))
        obj.set_angular_velocity(np.zeros(3))
        obj.set_visibility(False)
        self.occupied[slot] = False
        return {'elapsed': 0.}

    def observe(self, budget):
        self._recording_action = 'Tray scan'
        start, original_arm = self.sim_steps, self.arm
        self.limit_step = start + int(math.floor(budget/self.dt + 1e-8))
        confidence, timestamps, order, travel = np.zeros(8), np.full(8, -1.), scan_order(self.occupied, self.arm), 0.
        travel_intervals = []
        try:
            for tray in order:
                move_start, switching = self.sim_steps, tray != self.arm
                try:
                    self.move(self.views[tray], self.c['gripper_open'], self.c['normal_speed'])
                finally:
                    if switching:
                        travel += (self.sim_steps-move_start)*self.dt
                    travel_intervals.append([(move_start-start)*self.dt, (self.sim_steps-start)*self.dt])
                self.arm = tray
                rgb, pose = self.capture(), self.camera_pose()
                for slot in range(tray*4, tray*4+4):
                    if self.occupied[slot]:
                        confidence[slot] = self.detector.detect(rgb, self.pockets[slot], self.c['roi_radius'], pose)['confidence']
                        timestamps[slot] = self.sim_steps*self.dt
            # Restore the start station so measured scan cost and pick cost have a defined boundary.
            move_start, switching = self.sim_steps, original_arm != self.arm
            try:
                self.move(self.parks[original_arm], self.c['gripper_open'], self.c['normal_speed'])
            finally:
                if switching:
                    travel += (self.sim_steps-move_start)*self.dt
                travel_intervals.append([(move_start-start)*self.dt, (self.sim_steps-start)*self.dt])
            self.arm = original_arm
            return {'elapsed': (self.sim_steps-start)*self.dt, 'travel': travel,
                    'confidence': confidence.tolist(), 'timestamps': timestamps.tolist(),
                    'view_order': order, 'arm': self.arm, 'travel_intervals': travel_intervals}
        except BudgetExpired:
            return {'elapsed': (self.sim_steps-start)*self.dt, 'travel': travel,
                    'confidence': [0.]*8, 'interrupted': True, 'arm': self.arm, 'view_order': order, 'travel_intervals': travel_intervals}
        finally:
            self.limit_step = None

    def move(self, xyz, gripper, speed):
        initial = self.robot.get_joint_positions(joint_indices=self.indices)
        target = np.r_[self.kin.solve(xyz, seed=initial), gripper]
        duration = max(.05, float(np.max(np.abs(target-initial))) / speed)
        frames = int(math.ceil(duration/self.dt))
        for i in range(1, frames+1):
            s = i / frames
            smooth = s*s*(3-2*s)
            self.apply(initial + smooth*(target-initial))
            self.advance(self.dt)
        self.advance(self.c['settle_seconds'])
        if np.max(np.abs(self.robot.get_joint_positions(joint_indices=self.indices)[:5] - target[:5])) > self.c['motion_tolerance']:
            raise RuntimeError('Joint tracking exceeded configured tolerance')

    def grip(self, value):
        q = self.robot.get_joint_positions(joint_indices=self.indices)
        q[-1] = value
        self.apply(q)
        self.advance(self.c['close_seconds'])

    def pick(self, slot, mode, oracle, budget):
        self._recording_action = f'Pocket {slot} | {"careful" if mode else "normal"}'
        start = self.sim_steps
        self.limit_step = start + int(math.floor(budget/self.dt + 1e-8))
        tray = slot//4
        travel, confidence, dropped = 0., 0., False
        self.failure_reason = ''
        status, success = 'pick_failed', False
        try:
            # Nominal pocket coordinates define the view pose; RGB supplies the object offset.
            move_start, switching = self.sim_steps, tray != self.arm
            try:
                self.move([*self.pockets[slot, :2], self.c['view_z']], self.c['gripper_open'], self.c['normal_speed'])
            finally:
                if switching:
                    travel += (self.sim_steps-move_start)*self.dt
            self.arm = tray
            detection = self.sense(self.pockets[slot], self.c['roi_radius'])
            confidence = detection['confidence']
            if oracle:
                xy = self.objects[slot].get_world_pose()[0][:2]
            else:
                if confidence < self.c['confidence_threshold']:
                    raise DetectionFailed()
                xy = np.asarray(detection['xy'])
            self.move([*xy, self.c['reobserve_z']], self.c['gripper_open'], self.c['normal_speed'])
            # Re-localize from the new wrist pose immediately before descending to grasp.
            refinement = self.sense(self.pockets[slot], self.c['roi_radius'])
            if oracle:
                xy = self.objects[slot].get_world_pose()[0][:2]
            else:
                confidence = refinement['confidence']
                if confidence < self.c['confidence_threshold']:
                    raise DetectionFailed()
                xy = np.asarray(refinement['xy'])
            self.move([*xy, self.c['grasp_z']], self.c['gripper_open'], self.c['normal_speed'])
            self.grip(self.c['gripper_closed'])
            speed = self.c['normal_speed']/(self.c['careful_slow'] if mode else 1.)
            self.move([*xy, self.c['hover_z']], self.c['gripper_closed'], speed)
            # Privileged state below belongs to the evaluator; it is not used by camera localization.
            lifted = bool(self.objects[slot].get_world_pose()[0][2] > self.c['table_z']+.045)
            if lifted:
                output = self.outputs[tray]
                self.move([*output[:2], self.c['hover_z']], self.c['gripper_closed'], speed)
                dropped = bool(self.objects[slot].get_world_pose()[0][2] < self.c['table_z']+.035)
                self.move([*output[:2], self.c['grasp_z']], self.c['gripper_closed'], speed)
                self.grip(self.c['gripper_open'])
                self.move([*output[:2], self.c['view_z']], self.c['gripper_open'], self.c['normal_speed'])
                visual = self.sense(output, self.c['output_radius'])
                truth = self.objects[slot].get_world_pose()[0]
                in_output = np.linalg.norm(truth[:2]-output[:2]) < self.c['output_radius'] and abs(truth[2]-output[2]) < .015
                success = bool(in_output and (oracle or visual['confidence'] >= self.c['confidence_threshold']))
                status = 'completed' if success else ('dropped' if dropped else 'verify_failed')
        except DetectionFailed:
            status, success = 'detection_failed', False
        except BudgetExpired:
            status, success = 'interrupted', False
        except (RuntimeError, ValueError) as error:
            status, success = 'motion_failed', False
            self.failure_reason = str(error)
        # Recover to the defined park pose while preserving a failed object's physical position.
        if status != 'interrupted':
            try:
                self.grip(self.c['gripper_open'])
                self.move(self.parks[tray], self.c['gripper_open'], self.c['normal_speed'])
                self.arm = tray
            except BudgetExpired:
                status, success = 'interrupted', False
            except (RuntimeError, ValueError) as error:
                status, success = 'motion_failed', False
                self.failure_reason = str(error)
        self.limit_step = None
        return self.outcome(start, travel, success, dropped, status, confidence)

    def outcome(self, start, travel, success, dropped, status, confidence):
        return {'elapsed': (self.sim_steps-start)*self.dt, 'travel': float(travel),
                'success': bool(success), 'dropped': bool(dropped), 'status': status,
                'confidence': float(confidence), 'failure_reason': self.failure_reason}

connection = Client(('127.0.0.1', args.port), authkey=bytes.fromhex(args.auth))
cell = None
try:
    version = get_version()
    if tuple(str(x) for x in version[2:5]) != tuple(map(str, spec['runtime']['core_version'])):
        raise RuntimeError(f'This experiment targets Isaac Sim 6.0, received {version[0]}')
    cell = Workcell()
    import cv2
    runtime = {'isaac_version': list(version), 'target': spec['runtime'],
               'camera_api': 'isaacsim.sensors.experimental.rtx',
               'python': sys.version, 'numpy': np.__version__,
               'opencv': cv2.__version__, 'platform': platform.platform()}
    connection.send({'ok': True, 'result': {'calibration': cell.calibration,
                                         'joint_names': cell.robot.dof_names, 'runtime': runtime}})
    while True:
        request = connection.recv()
        operation, values = request['op'], request.get('values', {})
        if operation == 'close':
            connection.send({'ok': True, 'result': {}})
            break
        try:
            if operation == 'advance':
                before = cell.sim_steps
                cell.advance(values['seconds'])
                result = {'elapsed': (cell.sim_steps-before)*cell.dt}
            elif operation == 'frame':
                result = {'rgb': cell.capture(), 'timestamp': cell.sim_steps*cell.dt,
                          'camera_to_world_cv': cell.camera_pose().tolist()}
            else:
                result = getattr(cell, operation)(**values)
            connection.send({'ok': True, 'result': result})
        except Exception:
            connection.send({'ok': False, 'error': traceback.format_exc()})
except Exception:
    connection.send({'ok': False, 'error': traceback.format_exc()})
finally:
    try:
        if cell is not None:
            cell.abort_recording()
    finally:
        connection.close()
        application.close()


In [ ]:
class IsaacWorker:
    def __init__(self, timeout=300):
        command = Path(ISAAC_COMMAND[0]).expanduser()
        if not command.is_file():
            raise FileNotFoundError(f'Set ISAAC_COMMAND to the Isaac Sim 6.0 Python 3.12 command: {command}')
        if not WORKER_PATH.is_file() or not SHARED_MODULE_PATH.is_file():
            raise FileNotFoundError('Run the common-module and worker writefile cells first')
        auth = secrets.token_bytes(32)
        self.listener = Listener(('127.0.0.1', 0), authkey=auth)
        port = self.listener.address[1]
        self.connection, self.process = None, None
        worker_spec = {'cell': asdict(CELL), 'pockets': CELL.pockets().tolist(),
                       'views': CELL.views().tolist(), 'outputs': CELL.outputs().tolist(), 'parks': CELL.parks().tolist(),
                       'joint_names': list(ACTION_NAMES), 'usd': str(USD_PATH), 'urdf': str(URDF_PATH),
                       'headless': HEADLESS, 'runtime': ISAAC_RUNTIME}
        spec_path = RUN_DIR / 'worker_config.json'
        atomic_json(spec_path, worker_spec)
        self.log = (RUN_DIR / 'isaac_worker.log').open('w', encoding='utf-8')
        self.process = subprocess.Popen([*ISAAC_COMMAND, str(WORKER_PATH.resolve()),
                '--config', str(spec_path.resolve()), '--port', str(port), '--auth', auth.hex()],
                stdout=self.log, stderr=subprocess.STDOUT)
        # Accept in a bounded background future so failed startup cannot hang Jupyter.
        import queue
        import threading
        accepted = queue.Queue()
        def accept_connection():
            try:
                accepted.put((self.listener.accept(), None))
            except Exception as error:
                accepted.put((None, error))
        threading.Thread(target=accept_connection, daemon=True).start()
        try:
            deadline = time.monotonic() + timeout
            while accepted.empty():
                if self.process.poll() is not None or time.monotonic() >= deadline:
                    raise RuntimeError(f'Isaac startup failed/timed out. Read {RUN_DIR / "isaac_worker.log"}')
                time.sleep(.10)
            self.connection, error = accepted.get()
            if error is not None:
                raise error
            self.startup = self.receive(timeout)
        except Exception:
            self.close()
            raise
        finally:
            self.listener.close()
        atexit.register(self.close)

    def receive(self, timeout=300):
        if not self.connection.poll(timeout):
            raise TimeoutError(f'Isaac worker timed out. Read {RUN_DIR / "isaac_worker.log"}')
        reply = self.connection.recv()
        if not reply['ok']:
            raise RuntimeError(reply['error'])
        return reply['result']

    def call(self, operation, **values):
        if self.process.poll() is not None:
            raise RuntimeError(f'Isaac worker exited. Read {RUN_DIR / "isaac_worker.log"}')
        self.connection.send({'op': operation, 'values': values})
        return self.receive()

    def close(self):
        if self.connection is not None:
            try:
                if self.process is not None and self.process.poll() is None:
                    self.connection.send({'op': 'close'})
            except (EOFError, BrokenPipeError, OSError):
                pass
            self.connection.close()
            self.connection = None
        if self.process is not None and self.process.poll() is None:
            self.process.terminate()
            try:
                self.process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                self.process.kill()
                self.process.wait(timeout=10)
        if hasattr(self, 'listener'):
            self.listener.close()
        if hasattr(self, 'log'):
            self.log.close()

WORKER = IsaacWorker()
atomic_json(RUN_DIR / 'camera_calibration.json', WORKER.startup['calibration'])


## 5.1. Isaac sample pick

Detect one block with P2 mass using wrist RGB, pick it in normal mode, and move it to the output tray.
Use `SAMPLE_SLOT`, `SAMPLE_MODE`, and `SAMPLE_WEIGHT` to change the pocket, handling mode, and mass.
Inspect the pre-grasp wrist RGB image, the workcell overview / wrist RGB video, the outcome, and the elapsed time.
Results are saved in `results/isaac_sample/`. A failed pick stops execution before physical calibration.


In [ ]:
SAMPLE_SLOT = 0
SAMPLE_SEED = 40_000
SAMPLE_WEIGHT = PRODUCTS['P2'][0]
SAMPLE_MODE = 0  # 0: normal, 1: careful
SAMPLE_BUDGET_SECONDS = 60.
SAMPLE_DIR = RESULTS_DIR / 'isaac_sample'
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

WORKER.call('reset', seed=SAMPLE_SEED, arm=SAMPLE_SLOT//4, appearance='nominal')
try:
    WORKER.call('present', slot=SAMPLE_SLOT, weight=SAMPLE_WEIGHT, seed=SAMPLE_SEED)
    WORKER.call('start_recording', path=str((SAMPLE_DIR/'pick.mp4').resolve()), fps=15,
        label=f'Isaac sample | pocket {SAMPLE_SLOT}', eye=(.55, -.60, .45), target=(.10, 0., .08))
    WORKER.call('advance', seconds=CELL.feeder_seconds)
    WORKER.call('move', xyz=[*CELL.pockets()[SAMPLE_SLOT, :2], CELL.view_z],
                gripper=CELL.gripper_open, speed=CELL.normal_speed)
    sample_frame = WORKER.call('frame')
    sample_outcome = WORKER.call('pick', slot=SAMPLE_SLOT, mode=SAMPLE_MODE,
                                 oracle=False, budget=SAMPLE_BUDGET_SECONDS)
    sample_video = WORKER.call('stop_recording')
finally:
    try:
        WORKER.call('abort_recording')
    finally:
        WORKER.call('reset', seed=SAMPLE_SEED, arm=SAMPLE_SLOT//4, appearance='nominal')

plt.imsave(SAMPLE_DIR/'wrist_rgb.png', sample_frame['rgb'])
sample_result = {'purpose': 'isaac_sample', 'scene_seed': SAMPLE_SEED, 'slot': SAMPLE_SLOT,
    'weight': SAMPLE_WEIGHT, 'mode': 'careful' if SAMPLE_MODE else 'normal',
    'budget_seconds': SAMPLE_BUDGET_SECONDS, 'outcome': sample_outcome, 'video': sample_video,
    'wrist_rgb_path': str(SAMPLE_DIR/'wrist_rgb.png'), 'runtime': WORKER.startup['runtime']}
atomic_json(SAMPLE_DIR/'result.json', sample_result)


In [ ]:
from IPython.display import Video, display

plt.figure(figsize=(8, 6))
plt.imshow(sample_frame['rgb'])
plt.title('Wrist RGB before picking')
plt.axis('off')
plt.show()
display(pd.DataFrame([sample_result['outcome']])[
    ['status', 'success', 'elapsed', 'confidence', 'failure_reason']])
display(Video(filename=sample_result['video']['path'], embed=True, mimetype='video/mp4', width=960,
              html_attributes='controls preload="metadata"'))
if not sample_result['outcome']['success']:
    reason = sample_result['outcome']['failure_reason'] or sample_result['outcome']['status']
    raise RuntimeError(f'Isaac sample pick failed: {reason}. Inspect the sample video before calibration.')


## 6. Physical calibration before training

Calibration seeds are separate from training, validation, and test seeds. Measure continuous mass and handling-risk
values within the training range, excluding neighborhoods around P1–P4 and P5. Repeat each pocket×mode×station-switch
combination equally. Do not update this table using P5/P6 or appearance tests.

Failed objects are not returned to their original positions before another attempt. Independent calibration trials
reset the scene, while shift evaluation preserves each object's position after failure. The calibrated event environment
samples these physical trials. If every trial fails for a pocket, fix the geometry, gripper, or controller and repeat calibration.

Alongside the pick table `calibration.csv`, record scan time, station-switch time, and RGB confidence in
`observation.csv` for each starting-station×occupied-tray combination. The fast training environment also samples
these observations. The pick table includes the initial observation, approach re-observation, output verification,
and return time. Do not count time twice across the two tables.

Start/end intervals for station switches during observation are also saved. If a scan stops at the shift horizon,
the fast environment charges only the switching time elapsed up to that point.


In [ ]:
def sample_training_features(rng):
    if rng.random() < .5:
        return np.array(PRODUCTS[rng.choice(FAMILIAR)], dtype=float)
    low, high = np.array(ENV.feature_low), np.array(ENV.feature_high)
    held_out = np.array(PRODUCTS['P5'])
    while True:
        x = rng.uniform(low, high)
        if not np.all(np.abs(x-held_out) / (high-low) <= .10):
            return x

CALIBRATION_SPEC = SOURCE_SPEC | {'assets': ASSET_MANIFEST,
    'simulator_runtime': WORKER.startup['runtime'],
    'seed': CALIBRATION_SEED, 'repeats': CALIBRATION_REPEATS}
CALIBRATION_KEY = canonical_hash(CALIBRATION_SPEC)
CALIBRATION_PATH = RUN_DIR / 'calibration.csv'
CALIBRATION_META = RUN_DIR / 'calibration_meta.json'
OBSERVATION_PATH = RUN_DIR / 'observation.csv'

def measure_calibration():
    rng = np.random.default_rng(CALIBRATION_SEED)
    rows = []
    for slot in range(8):
        for mode in range(2):
            for switched in (0, 1):
                for repetition in range(CALIBRATION_REPEATS):
                    weight, fragility = sample_training_features(rng)
                    scene_seed = int(rng.integers(2**31))
                    arm = slot // 4 if not switched else 1-slot//4
                    WORKER.call('reset', seed=scene_seed, arm=arm, appearance='nominal')
                    WORKER.call('present', slot=slot, weight=float(weight), seed=scene_seed)
                    WORKER.call('advance', seconds=CELL.feeder_seconds)
                    outcome = WORKER.call('pick', slot=slot, mode=mode, oracle=False, budget=60.)
                    after = WORKER.call('observe', budget=60.)
                    rows.append({'slot': slot, 'mode': mode, 'switched': switched,
                        'repetition': repetition, 'scene_seed': scene_seed,
                        'weight': float(weight), 'fragility': float(fragility),
                        'next_confidence': after['confidence'][slot], **outcome})
        print(f'Calibration: pocket {slot+1}/8 completed')
    frame = pd.DataFrame(rows)
    observation_rows = []
    for arm in (0, 1):
        for tray_mask in (1, 2, 3):
            for repetition in range(CALIBRATION_REPEATS):
                scene_seed = int(rng.integers(2**31))
                WORKER.call('reset', seed=scene_seed, arm=arm, appearance='nominal')
                for slot in range(8):
                    if tray_mask & (1 << (slot//4)):
                        weight, _ = sample_training_features(rng)
                        WORKER.call('present', slot=slot, weight=float(weight), seed=scene_seed+slot)
                WORKER.call('advance', seconds=CELL.feeder_seconds)
                scan = WORKER.call('observe', budget=60.)
                if scan.get('interrupted') or scan['arm'] != arm:
                    raise RuntimeError('Wrist scan did not finish and restore the start station')
                observation_rows.append({'arm': arm, 'tray_mask': tray_mask,
                    'repetition': repetition, 'elapsed': scan['elapsed'], 'travel': scan['travel'],
                    'confidence': json.dumps(scan['confidence']), 'scene_seed': scene_seed,
                    'travel_intervals': json.dumps(scan['travel_intervals'])})
        print(f'Wrist observation calibration: start station {arm+1}/2 completed')
    observations = pd.DataFrame(observation_rows)
    frame.to_csv(CALIBRATION_PATH, index=False)
    observations.to_csv(OBSERVATION_PATH, index=False)
    atomic_json(CALIBRATION_META, {'key': CALIBRATION_KEY, 'spec': CALIBRATION_SPEC,
        'data_hash': file_hash(CALIBRATION_PATH), 'observation_data_hash': file_hash(OBSERVATION_PATH)})
    return frame, observations

if CALIBRATION_PATH.exists() and CALIBRATION_META.exists() and OBSERVATION_PATH.exists():
    saved = json.loads(CALIBRATION_META.read_text())
    if (saved['key'] != CALIBRATION_KEY or saved['data_hash'] != file_hash(CALIBRATION_PATH)
            or saved['observation_data_hash'] != file_hash(OBSERVATION_PATH)):
        raise RuntimeError('Calibration provenance changed. Use a fresh RUN_DIR or remove the old calibration files.')
    calibration_df = pd.read_csv(CALIBRATION_PATH)
    observation_df = pd.read_csv(OBSERVATION_PATH)
else:
    if any(p.exists() for p in (CALIBRATION_PATH, CALIBRATION_META, OBSERVATION_PATH)):
        raise RuntimeError('Incomplete/old calibration. Use a fresh RUN_DIR for the wrist-camera experiment.')
    calibration_df, observation_df = measure_calibration()
if not calibration_df['elapsed'].gt(0).all():
    raise ValueError('Pick measurements must have positive elapsed times')
if calibration_df['status'].eq('interrupted').any():
    raise RuntimeError('Calibration contains interrupted picks; repeat those measurements')
success_by_pocket = calibration_df[calibration_df['mode'].eq(0)].groupby('slot')['success'].sum()
if len(success_by_pocket) != 8 or (success_by_pocket < 1).any():
    raise RuntimeError('At least one pocket has no successful normal pick. Fix the executor before training.')
display(calibration_df.groupby(['mode', 'switched']).agg(
    attempts=('success', 'size'), success_rate=('success', 'mean'),
    drop_rate=('dropped', 'mean'), mean_seconds=('elapsed', 'mean')))

display(observation_df.groupby(['arm', 'tray_mask'])[['elapsed', 'travel']].mean())


## 7. Measurement-based training model and shared shifts

The fast environment bootstraps measurement rows with similar mass and risk from the same pocket, mode,
and station-switch context. Mean time, standard deviation, and failure rate are used only in observations
and rule-based mode selection. These results do not replace physical experiments. Save both surrogate and
RGB+physics results for the same policies to assess model bias. Predictions for OOD P6 do not establish extrapolation.

Each shift uses the same arrival, order, and scene seeds across policies. Physical failures are not forced to match.
Event-environment bootstrap draws are keyed by job, attempt, and action, independently of policy execution order.

Tray scans before each decision use a separate measurement model. The fast environment also accounts for orders
arriving during observation motion and their lateness. Sensor capture latency alone does not replace scan time.


In [ ]:
class MeasuredModel:
    def __init__(self, frame):
        required = {'slot', 'mode', 'switched', 'weight', 'fragility', 'elapsed', 'travel',
                    'success', 'dropped', 'status', 'confidence', 'next_confidence'}
        if not required <= set(frame.columns):
            raise ValueError(f'Missing calibration columns: {required-set(frame.columns)}')
        self.frame = frame.reset_index(drop=True).copy()
        for name in ('success', 'dropped'):
            if self.frame[name].dtype == object:
                self.frame[name] = self.frame[name].astype(str).str.lower().map({'true': True, 'false': False})
        if self.frame[list(required)].isna().any().any():
            raise ValueError('Invalid calibration rows')
        self.groups = {}
        for slot in range(8):
            for mode in range(2):
                for switched in (0, 1):
                    values = self.frame.query('slot == @slot and mode == @mode and switched == @switched')
                    if len(values) < 4:
                        raise ValueError(f'Need at least four measurements for {(slot, mode, switched)}')
                    self.groups[slot, mode, switched] = values
        self.cache = {}

    def neighbors(self, weight, fragility, slot, mode, switched):
        group = self.groups[int(slot), int(mode), int(switched)]
        xy = group[['weight', 'fragility']].to_numpy()
        distance = np.sum(((xy-np.array([weight, fragility])) / np.array([.18, .9]))**2, axis=1)
        return group.iloc[np.argsort(distance, kind='stable')[:min(8, len(group))]]

    def predict(self, weight, fragility, slot, mode=0, switched=0):
        key = (round(float(weight), 4), round(float(fragility), 3), int(slot), int(mode), int(switched))
        if key not in self.cache:
            rows = self.neighbors(*key)
            self.cache[key] = {'seconds': float(rows['elapsed'].mean()),
                'std': float(rows['elapsed'].std(ddof=0)), 'travel': float(rows['travel'].mean()),
                'drop_probability': float(rows['dropped'].mean()),
                'failure_probability': float(1-rows['success'].mean())}
        return self.cache[key]

    def draw(self, weight, fragility, slot, mode, switched, seed):
        rows = self.neighbors(weight, fragility, slot, mode, switched)
        rng = np.random.default_rng(np.random.SeedSequence(seed))
        return rows.iloc[int(rng.integers(len(rows)))].to_dict()

class MeasuredScanModel:
    def __init__(self, frame):
        required = {'arm', 'tray_mask', 'elapsed', 'travel', 'confidence', 'travel_intervals'}
        if not required <= set(frame.columns) or frame[list(required)].isna().any().any():
            raise ValueError('Missing/invalid wrist observation measurements')
        self.groups = {}
        for arm in (0, 1):
            for tray_mask in (1, 2, 3):
                rows = frame.query('arm == @arm and tray_mask == @tray_mask').copy()
                if len(rows) < 4 or (rows['elapsed'] <= 0).any() or (rows['travel'] < 0).any() or (rows['travel'] > rows['elapsed']).any():
                    raise ValueError(f'Invalid wrist scan measurements for {(arm, tray_mask)}')
                for value in rows['confidence']:
                    confidence = np.asarray(json.loads(value), dtype=float)
                    if confidence.shape != (8,) or not np.isfinite(confidence).all() or ((confidence < 0) | (confidence > 1)).any():
                        raise ValueError('Invalid scan confidence vector')
                matrix = np.asarray([json.loads(value) for value in rows['confidence']])
                required_slots = np.array([bool(tray_mask & (1 << (slot//4))) for slot in range(8)])
                if (matrix[:, required_slots].max(axis=0) < CELL.confidence_threshold).any():
                    raise ValueError(f'At least one pocket is never visible in scan {(arm, tray_mask)}')
                for row in rows.itertuples():
                    intervals = np.asarray(json.loads(row.travel_intervals), dtype=float).reshape(-1, 2)
                    if (not np.isfinite(intervals).all() or (intervals < 0).any()
                            or (intervals[:, 1] < intervals[:, 0]).any()
                            or (intervals[:, 1] > row.elapsed+1e-7).any()
                            or (len(intervals)>1 and (intervals[1:, 0] < intervals[:-1, 1]-1e-7).any())
                            or abs((intervals[:, 1]-intervals[:, 0]).sum()-row.travel)>1e-7):
                        raise ValueError('Invalid measured scan travel intervals')
                self.groups[arm, tray_mask] = rows

    def draw(self, arm, tray_mask, seed):
        rows = self.groups[int(arm), int(tray_mask)]
        rng = np.random.default_rng(np.random.SeedSequence(seed))
        row = rows.iloc[int(rng.integers(len(rows)))].to_dict()
        row['confidence'] = json.loads(row['confidence'])
        row['travel_intervals'] = json.loads(row['travel_intervals'])
        return row

    def seconds(self, arm, tray_mask):
        return float(self.groups[int(arm), int(tray_mask)]['elapsed'].mean())

MODEL = MeasuredModel(calibration_df)
SCAN_MODEL = MeasuredScanModel(observation_df)
def make_experiment_key():
    return canonical_hash({'experiment_version': EXPERIMENT_VERSION, 'settings': SOURCE_SPEC,
        'calibration': CALIBRATION_KEY, 'data_hash': file_hash(CALIBRATION_PATH),
        'observation_hash': file_hash(OBSERVATION_PATH)})

EXPERIMENT_KEY = make_experiment_key()

@dataclass(frozen=True)
class ShiftInstance:
    arrival: np.ndarray
    deadline: np.ndarray
    weight: np.ndarray
    fragility: np.ndarray
    tray: np.ndarray
    scene_seed: int
    appearance: str = 'nominal'

    def __len__(self):
        return len(self.arrival)

def sample_instance(cfg, rng, products=None, arrival_rate=None, appearance='nominal'):
    rate = cfg.arrival_rate if arrival_rate is None else arrival_rate
    count = int(rng.poisson(rate * cfg.shift_length))
    arrival = np.sort(np.r_[np.zeros(cfg.initial_jobs), rng.uniform(0, cfg.shift_length, count)])
    features = np.array([sample_training_features(rng) if products is None
                         else PRODUCTS[rng.choice(products)] for _ in arrival])
    tray = rng.integers(2, size=len(arrival))
    # Due-date scale is frozen from training calibration and never fitted on test attempts.
    nominal = np.array([MODEL.predict(w, f, int(t)*4)['seconds'] + SCAN_MODEL.seconds(0, 1 << int(t))
                        for (w, f), t in zip(features, tray)])
    deadline = arrival + nominal + rng.uniform(*cfg.deadline_slack, len(arrival))
    return ShiftInstance(arrival, deadline, features[:, 0], features[:, 1], tray,
                         int(rng.integers(2**31)), appearance)

def make_instances(cfg, products, count, seed, appearance='nominal'):
    return [sample_instance(cfg, np.random.default_rng(seed+i), products,
                            cfg.arrival_rate, appearance) for i in range(count)]

class EventBackend:
    name = 'surrogate'
    def __init__(self, model=MODEL, scan_model=SCAN_MODEL):
        self.model, self.scan_model = model, scan_model

    def reset(self, instance, arm=0):
        self.inst, self.arm = instance, arm
        self.jobs = {}
        self.scan_index = 0
        self.confidence = np.zeros(8)

    def present(self, slot, job):
        self.jobs[slot] = job
        self.confidence[slot] = 0.  # A feeder event is not an RGB observation.

    def remove(self, slot):
        self.jobs.pop(slot, None)
        self.confidence[slot] = 0.

    def advance(self, seconds):
        return {'elapsed': seconds}

    def observe(self, budget):
        occupied = np.array([slot in self.jobs for slot in range(8)])
        tray_mask = sum(1 << tray for tray in (0, 1) if np.any(occupied[tray*4:tray*4+4]))
        if not tray_mask:
            return {'elapsed': 0., 'travel': 0., 'confidence': [0.]*8, 'arm': self.arm}
        row = self.scan_model.draw(self.arm, tray_mask, [self.inst.scene_seed, self.scan_index, 431])
        self.scan_index += 1
        if row['elapsed'] > budget:
            travel = sum(max(0., min(end, budget)-start) for start, end in row['travel_intervals'])
            return {'elapsed': budget, 'travel': travel, 'confidence': [0.]*8,
                    'interrupted': True, 'arm': self.arm}
        self.confidence = np.asarray(row['confidence'])*occupied
        return {'elapsed': row['elapsed'], 'travel': row['travel'],
                'confidence': self.confidence.tolist(), 'arm': self.arm,
                'view_order': scan_order(occupied, self.arm), 'travel_intervals': row['travel_intervals']}

    def pick(self, slot, mode, attempt, budget):
        job = self.jobs[slot]
        row = self.model.draw(self.inst.weight[job], self.inst.fragility[job], slot, mode,
            slot//4 != self.arm, [self.inst.scene_seed, job, attempt, slot, mode, int(slot//4 != self.arm)])
        self.arm = slot//4
        if row['elapsed'] > budget:
            return {'elapsed': budget, 'travel': min(row['travel'], budget), 'success': False,
                    'dropped': False, 'status': 'interrupted', 'confidence': row['confidence']}
        if not row['success']:
            self.confidence[slot] = row['next_confidence']
        return {k: row[k] for k in ('elapsed', 'travel', 'success', 'dropped', 'status', 'confidence')}

class PhysicsBackend:
    name = 'physics_camera'
    def __init__(self, worker, oracle=False):
        self.worker, self.oracle = worker, oracle
        if oracle:
            self.name = 'physics_oracle'

    def reset(self, instance, arm=0):
        self.inst, self.jobs = instance, {}
        self.worker.call('reset', seed=instance.scene_seed, arm=arm, appearance=instance.appearance)

    def present(self, slot, job):
        self.jobs[slot] = job
        scene_seed = int(np.random.SeedSequence([self.inst.scene_seed, job]).generate_state(1)[0])
        self.worker.call('present', slot=slot, weight=float(self.inst.weight[job]), seed=scene_seed)

    def remove(self, slot):
        self.jobs.pop(slot, None)
        self.worker.call('remove', slot=slot)

    def advance(self, seconds):
        return self.worker.call('advance', seconds=seconds)

    def observe(self, budget):
        result = self.worker.call('observe', budget=budget)
        if self.oracle:
            result['confidence'] = [float(i in self.jobs) for i in range(8)]
        return result

    def pick(self, slot, mode, attempt, budget):
        return self.worker.call('pick', slot=slot, mode=mode, oracle=self.oracle, budget=budget)


## 8. Scheduling environment and observation dimensions

The default `OBSERVATION_PROFILE='scheduling'` uses **8 jobs × 9 values + 7 global values = 79 dimensions**.
The RGB executor remains active; the network receives structured features, not the full RGB image.
The optional `OBSERVATION_PROFILE='vision'` adds RGB confidence and processing-time standard deviation
for each job, giving **8 × 11 + 7 = 95 dimensions**. These additions are not required for camera use,
and their benefit has not been established.

| Input | Count | Purpose |
|---|---:|---|
| Presence, processing time, mass, handling risk, deadline slack, tray A/B, waiting time, retry status | 9 per job | Original scheduling inputs |
| RGB confidence | 1 per job, optional | Confidence in the current detection |
| Processing-time standard deviation | 1 per job, optional | Uncertainty in measured execution time |
| Remaining time, arm A/B, backlog, three reward weights | 7 global | Workcell state and costs |

Both profiles use the same RGB executor, observation time, action masks, and reward.
Use the 95-dimensional profile only for an explicit comparison against the 79-dimensional default.
Each profile has a separate settings hash and output path to prevent mixing weights.

There are 16 actions, encoded as `mode * 8 + pocket`. Orders waiting to be supplied remain in each tray's FIFO queue.
Masks allow only occupied pockets confirmed by RGB. If all pockets are hidden, scan again. Jobs that remain undetected
are removed with failure and scrap costs.

The reward is on-time completion +1 − accumulated lateness − station-switch time cost − physical drop cost
− other pick-failure cost − scrap cost − unfinished cost at the horizon. Drop and other failure costs are not double-counted.
Observation motion, settling, capture time, and station-switch costs are included.
`sensor` is the total scan time; `observation_travel` is the station-switch time within that scan.


In [ ]:
class LVHMSchedulingEnv(gym.Env):
    metadata = {'render_modes': []}
    def __init__(self, config=None, backend=None):
        super().__init__()
        self.cfg = config or ENV
        self.backend = backend or EventBackend()
        self.slot_dim, self.global_dim = OBS_SCHEMA['slot_dim'], OBS_SCHEMA['global_dim']
        self.num_modes = 2 if self.cfg.careful_mode else 1
        self.action_space = spaces.Discrete(8*self.num_modes)
        low = np.zeros(OBS_SCHEMA['total_dim'], dtype=np.float32)
        low[np.arange(8)*self.slot_dim+4] = -1.
        high = np.ones(OBS_SCHEMA['total_dim'], dtype=np.float32)
        high[8*self.slot_dim+3] = self.cfg.max_queue_obs/8
        high[-3:] = 2.
        self.observation_space = spaces.Box(low, high, dtype=np.float32)
        self._done = True

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        options = options or {}
        rate = options.get('arrival_rate')
        if rate is None and self.cfg.arrival_rate_range is not None:
            rate = float(self.np_random.uniform(*self.cfg.arrival_rate_range))
        self._inst = options.get('instance')
        if self._inst is None:
            self._inst = sample_instance(self.cfg, self.np_random, options.get('products'), rate)
        if len(self._inst) < 1:
            raise ValueError('A shift needs at least one initial job')
        self._w = np.asarray(options.get('weights', self.np_random.uniform(self.cfg.weight_low, self.cfg.weight_high)))
        self._attempts = np.zeros(len(self._inst), dtype=int)
        self._slots = np.full(8, -1, dtype=int)
        self._pending = [deque(), deque()]
        self._remaining = set(range(len(self._inst)))
        self._next, self._t, self._arm, self._done = 0, 0., 0, False
        self._confidence = np.zeros(8)
        self._events = []
        self._stats = {k: 0 for k in ('decisions', 'completed', 'on_time', 'late', 'drops',
            'failures', 'detection_failures', 'scrapped', 'unfinished', 'careful')}
        self._stats.update(tardiness=0., travel=0., score=0., feeder=0., sensor=0., observation_travel=0.,
                           solver_calls=0, solver_optimal=0, solver_fallback=0, planning_seconds=0.)
        self.backend.reset(self._inst)
        self._admit()
        # Initial feeder/camera overhead is charged at the first transition, not discarded.
        self._startup_reward = self._prepare()
        if self._done:
            self._stats['score'] = float(self._startup_reward)
        return self._obs(), {'t': self._t, 'backend': self.backend.name}

    def _admit(self):
        inst = self._inst
        while self._next < len(inst) and inst.arrival[self._next] <= self._t + 1e-9:
            job = self._next
            self._pending[int(inst.tray[job])].append(job)
            self._next += 1

    def _tick(self, elapsed):
        if elapsed < 0 or elapsed > self.cfg.shift_length-self._t + 1e-7:
            raise ValueError('Backend elapsed time exceeds the remaining shift budget')
        start, end = self._t, min(self.cfg.shift_length, self._t+elapsed)
        indices = np.array(sorted(self._remaining), dtype=int)
        inst = self._inst
        # Includes jobs arriving while the arm is busy and jobs still waiting at the feeder.
        late = float(np.maximum(0., end-np.maximum.reduce(
            [np.full(len(indices), start), inst.arrival[indices], inst.deadline[indices]])).sum())
        self._stats['tardiness'] += late
        self._t = end
        self._admit()
        return -self._w[0]*late

    def _advance(self, seconds):
        dt = min(seconds, max(0., self.cfg.shift_length-self._t))
        # Cap at a whole physics step; any final sub-step is accounted for logically.
        physical_dt = math.floor((dt+1e-9)/CELL.physics_dt)*CELL.physics_dt
        if physical_dt > 0:
            actual = self.backend.advance(physical_dt)['elapsed']
            if abs(actual-physical_dt) > 1e-7:
                raise RuntimeError('Backend clock mismatch')
        return self._tick(dt)

    def _scrap(self, slot, reason):
        job = int(self._slots[slot])
        self._remaining.remove(job)
        self._slots[slot] = -1
        self.backend.remove(slot)
        self._stats['scrapped'] += 1
        self._events.append({'t': self._t, 'job': job, 'slot': slot, 'status': reason})
        # Removal is the external recovery/collection operation, with an explicit time cost.
        return -self.cfg.scrap_penalty + self._advance(CELL.clearance_seconds)

    def _finish(self):
        self._done = True
        unfinished = int(sum(self._inst.arrival[j] < self.cfg.shift_length for j in self._remaining))
        self._stats['unfinished'] = unfinished
        return -self.cfg.unfinished_penalty*unfinished

    def _prepare(self):
        reward = 0.
        recaptures = 0
        while self._t < self.cfg.shift_length - 1e-8:
            self._admit()
            new_count = self._fill_pockets()
            if new_count:
                duration = min(CELL.feeder_seconds*new_count, self.cfg.shift_length-self._t)
                self._stats['feeder'] += duration
                reward += self._advance(duration)
            if self._t >= self.cfg.shift_length - 1e-8:
                break
            if np.any(self._slots >= 0):
                scan_start = self._t
                measurement = self.backend.observe(self.cfg.shift_length-self._t)
                self._stats['sensor'] += measurement['elapsed']
                reward += self._tick(measurement['elapsed'])
                travel = float(measurement.get('travel', 0.))
                if travel < 0 or travel > measurement['elapsed']+1e-7:
                    raise ValueError('Invalid observation travel time')
                self._stats['travel'] += travel
                self._stats['observation_travel'] += travel
                reward -= self._w[1]*travel
                self._arm = int(measurement.get('arm', self._arm))
                self._events.append({'status': 'observe', 'start': scan_start, 'end': self._t, **measurement})
                self._confidence = np.asarray(measurement['confidence'])
                if measurement.get('interrupted') or self._t >= self.cfg.shift_length - 1e-8:
                    reward += self._advance(self.cfg.shift_length-self._t)
                    break
                if self.action_masks().any():
                    return reward
                recaptures += 1
                if recaptures >= CELL.max_recaptures:
                    for slot in np.flatnonzero(self._slots >= 0):
                        if self._t >= self.cfg.shift_length - 1e-8:
                            break
                        self._stats['failures'] += 1
                        self._stats['detection_failures'] += 1
                        reward -= self.cfg.failure_penalty
                        reward += self._scrap(int(slot), 'unobservable')
                    recaptures = 0
                continue
            if self._next < len(self._inst):
                reward += self._advance(max(0., self._inst.arrival[self._next]-self._t))
            else:
                reward += self._advance(self.cfg.shift_length-self._t)
        return reward + self._finish()

    def _fill_pockets(self):
        new_count = 0
        for slot in range(8):
            tray = slot//4
            if self._slots[slot] < 0 and self._pending[tray]:
                job = self._pending[tray].popleft()
                self._slots[slot] = job
                self.backend.present(slot, job)
                new_count += 1
        return new_count

    def action_masks(self):
        valid = (self._slots >= 0) & (self._confidence >= CELL.confidence_threshold)
        return np.tile(valid & (not self._done), self.num_modes)

    def step(self, action):
        if self._done or not self.action_space.contains(action) or not self.action_masks()[action]:
            raise ValueError(f'Invalid or masked action: {action}')
        slot, mode = int(action)%8, int(action)//8
        job = int(self._slots[slot])
        start = self._t
        outcome = self.backend.pick(slot, mode, int(self._attempts[job]), self.cfg.shift_length-self._t)
        if outcome['elapsed'] <= 0:
            raise RuntimeError('A pick must advance the simulation clock')
        reward = self._startup_reward + self._tick(float(outcome['elapsed']))
        self._startup_reward = 0.
        travel = float(outcome['travel'])
        reward -= self._w[1]*travel
        self._stats['travel'] += travel
        self._arm = slot//4
        interrupted = outcome['status'] == 'interrupted'
        if outcome['dropped']:
            self._stats['drops'] += 1
            reward -= self._w[2]
        if not interrupted:
            if outcome['success']:
                self._remaining.remove(job)
                self._slots[slot] = -1
                self.backend.remove(slot)
                self._stats['completed'] += 1
                if self._t <= self._inst.deadline[job]:
                    self._stats['on_time'] += 1
                    reward += self.cfg.on_time_bonus
                else:
                    self._stats['late'] += 1
                reward += self._advance(CELL.clearance_seconds)
            else:
                self._attempts[job] += 1
                self._stats['failures'] += 1
                if not outcome['dropped']:
                    reward -= self.cfg.failure_penalty
                if outcome['status'] == 'detection_failed':
                    self._stats['detection_failures'] += 1
                if self._attempts[job] >= self.cfg.max_attempts:
                    reward += self._scrap(slot, 'attempts_exhausted')
        self._stats['decisions'] += 1
        self._stats['careful'] += mode
        self._events.append({'job': job, 'slot': slot, 'mode': mode, 'start': start,
                             'end': self._t, **outcome})
        if interrupted:
            reward += self._advance(self.cfg.shift_length-self._t) + self._finish()
        else:
            reward += self._prepare()
        self._stats['score'] += float(reward)
        info = {'t': self._t, 'outcome': outcome}
        if self._done:
            info['episode_metrics'] = dict(self._stats)
        return self._obs(), float(reward), self._done, False, info

    def _obs(self):
        slots = np.zeros((8, self.slot_dim), dtype=np.float32)
        cfg, inst = self.cfg, self._inst
        for slot, job in enumerate(self._slots):
            if job < 0:
                continue
            est = MODEL.predict(inst.weight[job], inst.fragility[job], slot, 0, slot//4 != self._arm)
            slots[slot] = [1., min(est['seconds']/cfg.proc_scale, 1.),
                min(inst.weight[job]/cfg.weight_scale, 1.), inst.fragility[job],
                np.clip((inst.deadline[job]-self._t)/cfg.deadline_scale, -1., 1.),
                float(slot//4 == 0), float(slot//4 == 1),
                min((self._t-inst.arrival[job])/cfg.wait_scale, 1.),
                float(self._attempts[job] > 0), self._confidence[slot], min(est['std']/cfg.proc_scale, 1.)][:self.slot_dim]
        backlog = sum(inst.arrival[j] <= self._t for j in self._remaining)
        global_obs = np.r_[max(0., 1-self._t/cfg.shift_length), np.eye(2)[self._arm],
            min(backlog, cfg.max_queue_obs)/8,
            np.clip((self._w-np.array(cfg.weight_low))/(np.array(cfg.weight_high)-cfg.weight_low), 0., 2.)]
        return np.r_[slots.ravel(), global_obs].astype(np.float32)


## 9. Rule policies and shared evaluation code

Compare Random, FCFS, SPT, LRF, EDF, and NTF in both normal-only and mode-enabled variants.
Mode-enabled rules compare expected lateness, travel, drop, and failure costs using the same measurement model.
The primary comparison uses the six mode-enabled rules, ATCSR+C, and Rolling CP-SAT.
RL and rules share observations, masks, the executor, and the reward. Future orders are supplied only to the
clairvoyant reference calculation.


In [ ]:
def valid_slots(env):
    return np.flatnonzero(env.action_masks()[:8])

def choose_mode(env, slot):
    if env.num_modes == 1:
        return 0
    job, inst = env._slots[slot], env._inst
    late_count = sum(inst.deadline[j] <= env._t for j in env._remaining if inst.arrival[j] <= env._t)
    costs = []
    alpha, beta, delta = env._w
    for mode in range(2):
        est = MODEL.predict(inst.weight[job], inst.fragility[job], slot, mode, slot//4 != env._arm)
        expected_late = max(0., env._t+est['seconds']-max(env._t, inst.deadline[job]))
        costs.append(alpha*(late_count*est['seconds']+expected_late) + beta*est['travel']
            + delta*est['drop_probability'] + env.cfg.failure_penalty*est['failure_probability'])
    return int(np.argmin(costs))

def rule_baselines():
    policies = {}
    for name in ('Random', 'FCFS', 'SPT', 'LRF', 'EDF', 'NTF'):
        for modes in (False, True):
            def policy(obs, env, rule=name, allow_modes=modes):
                slots = valid_slots(env)
                if not len(slots):
                    raise RuntimeError('Environment returned an empty mask at a decision point')
                inst = env._inst
                def key(slot):
                    job = env._slots[slot]
                    if rule == 'FCFS':
                        return (inst.arrival[job], job)
                    if rule == 'SPT':
                        return (MODEL.predict(inst.weight[job], inst.fragility[job], slot, 0,
                                              slot//4 != env._arm)['seconds'], job)
                    if rule == 'LRF':
                        return (inst.fragility[job], job)
                    if rule == 'EDF':
                        return (inst.deadline[job], job)
                    return (slot//4 != env._arm, inst.arrival[job], job)
                if rule == 'Random':
                    rng = np.random.default_rng(np.random.SeedSequence([inst.scene_seed, env._stats['decisions'], 991]))
                    slot = int(rng.choice(slots))
                else:
                    slot = int(min(slots, key=key))
                mode = choose_mode(env, slot) if allow_modes else 0
                return mode*8+slot
            policies[name+('_modes' if modes else '_normal')] = policy
    policies.update({'ATCSR+C': TUNED_ATCSR, 'Rolling CP-SAT': rolling_cp_sat})
    return policies

def edf(obs, env):
    slot = min(valid_slots(env), key=lambda s: (env._inst.deadline[env._slots[s]], env._slots[s]))
    return int(slot)

METRICS = ['on_time', 'completed', 'tardiness', 'travel', 'drops', 'failures',
           'detection_failures', 'scrapped', 'unfinished', 'careful', 'sensor', 'observation_travel', 'score',
           'solver_calls', 'solver_optimal', 'solver_fallback', 'planning_seconds']

def run_episode(env, policy, instance, weights=EVAL_WEIGHTS):
    obs, _ = env.reset(options={'instance': instance, 'weights': weights})
    if env._done:
        return dict(env._stats)
    for _ in range(10000):
        obs, _, terminated, truncated, info = env.step(policy(obs, env))
        if terminated or truncated:
            return info['episode_metrics']
    raise RuntimeError('Episode exceeded its bounded decision count')

def evaluate(policy, instances, cfg=ENV, weights=EVAL_WEIGHTS, backend_factory=None):
    env = LVHMSchedulingEnv(cfg, backend_factory() if backend_factory else EventBackend())
    try:
        return [run_episode(env, policy, inst, weights) for inst in instances]
    finally:
        env.close()

def mean_metrics(results):
    return {m: float(np.mean([r[m] for r in results])) for m in METRICS}

class Validator:
    # Surrogate validation nominates up to three candidates; physics validation chooses among them.
    def __init__(self, out_dir):
        self.instances = make_instances(ENV, FAMILIAR, N_VAL_SHIFTS, VAL_SEED)
        self.out_dir, self.best, self.rows, self.candidates = out_dir, -np.inf, [], []
        log_path, candidates_path = out_dir/'validation.csv', out_dir/'candidates.json'
        if log_path.exists():
            self.rows = pd.read_csv(log_path).to_dict('records')
            self.best = max(row['score'] for row in self.rows)
        if candidates_path.exists():
            self.candidates = json.loads(candidates_path.read_text())

    def __call__(self, policy, timesteps, save_best):
        import shutil
        row = {'timesteps': timesteps} | mean_metrics(evaluate(policy, self.instances))
        if not self.rows or self.rows[-1]['timesteps'] != timesteps:
            self.rows.append(row)
        self.best = max(self.best, row['score'])
        if len(self.candidates) < 3 or row['score'] > min(x['score'] for x in self.candidates):
            save_best()
            source = next(self.out_dir.glob('best_model.*'))
            name = f'candidate_{timesteps}{source.suffix}'
            shutil.copyfile(source, self.out_dir / name)
            self.candidates = [x for x in self.candidates if x['timesteps'] != timesteps]
            self.candidates.append({'timesteps': timesteps, 'score': row['score'], 'file': name})
            self.candidates.sort(key=lambda x: (-x['score'], x['timesteps']))
            for discarded in self.candidates[3:]:
                (self.out_dir/discarded['file']).unlink(missing_ok=True)
            self.candidates = self.candidates[:3]
            shutil.copyfile(self.out_dir/self.candidates[0]['file'], source)
            atomic_json(self.out_dir / 'candidates.json', self.candidates)
        pd.DataFrame(self.rows).to_csv(self.out_dir / 'validation.csv', index=False)
        print(f'  step {timesteps:>7}  surrogate validation score {row["score"]:.2f}')


### 9.1. Strong baselines and CP-SAT references

The eight primary baselines are the six `_modes` rules, **ATCSR+C**, and **Rolling CP-SAT**.
The six `_normal` rules remain auxiliary controls for the effect of careful-mode selection.
ATCSR+C's k1/k2 parameters are selected only on familiar validation shifts. All currently visible jobs are ready,
so the ready-time term is 1; this implementation is effectively ATCS+C.

Rolling CP-SAT uses valid pockets, executes only the first action, and solves again. It uses an **approximate model
of the current queue** with mean durations, observation travel, and expected failure costs. It does not read future orders.
`OPTIMAL` describes this approximate model, not optimality in the physical or stochastic environment.
Solver status, gap, planning time, and ATCSR fallback counts are saved for time-limited solves.

Clairvoyant CP-SAT knows future orders and solves an optimistic problem using minimum measured durations,
with travel, perception, failure, and pocket constraints removed. It maximizes only the positive on-time completion
bonus, so the solver bound is an **upper bound on the calibrated event-environment score**. An incumbent is not used
as the upper bound when optimality is unproven. There is no evidence that the minimum measured duration is a lower
bound for actual PhysX execution, so the result is not reported as a physical-score CP-SAT bound or physical headroom.
`physics_oracle` is a ground-truth-position control, distinct from a future-arrival oracle.


In [ ]:
def atcsr_policy(k1=1.0, k2=1.0):
    # Ready-time discounts equal one: every selectable pocket already contains a job.
    def policy(obs, env):
        slots = valid_slots(env)
        inst = env._inst
        estimates = {int(slot): MODEL.predict(inst.weight[env._slots[slot]],
            inst.fragility[env._slots[slot]], int(slot), 0, slot//4 != env._arm) for slot in slots}
        setup = {int(slot): max(0., estimates[int(slot)]['seconds'] - MODEL.predict(
            inst.weight[env._slots[slot]], inst.fragility[env._slots[slot]], int(slot), 0, 0)['seconds'])
            for slot in slots}
        mean_p = max(1e-6, np.mean([value['seconds'] for value in estimates.values()]))
        mean_s = max(1e-6, np.mean(list(setup.values())))
        def rank(slot):
            job = env._slots[slot]
            duration = max(1e-6, estimates[int(slot)]['seconds'])
            slack = max(0., inst.deadline[job]-env._t-duration)
            score = np.exp(-slack/(k1*mean_p)-setup[int(slot)]/(k2*mean_s))/duration
            return (-score, inst.arrival[job], job)
        slot = int(min(slots, key=rank))
        return choose_mode(env, slot)*8+slot
    return policy


def solve_queue_plan(jobs, duration, travel, loss, scan_seconds, scan_travel,
                     start_arm, now, horizon, weights, bonus=1., unfinished=1.,
                     time_limit=1., time_scale=100, cost_scale=100_000):
    """Solve a deterministic, one-attempt approximation of the visible queue.

    duration/travel/loss have shape (jobs, modes, switched). Hidden arrivals,
    retries, feeder occupancy and stochastic contacts are not predicted here.
    """
    from ortools.sat.python import cp_model
    if not jobs:
        raise ValueError('A queue plan needs at least one visible job')
    n, modes, _ = np.asarray(duration).shape
    if n != len(jobs) or np.asarray(travel).shape != (n, modes, 2) or np.asarray(loss).shape != (n, modes, 2):
        raise ValueError('Planner arrays must match the visible jobs and modes')
    if not np.isfinite([now, horizon, *weights, bonus, unfinished]).all() or now > horizon:
        raise ValueError('Invalid planning clock or weights')
    tick = lambda value: int(round(float(value)*time_scale))
    t0, end_shift = tick(now), tick(horizon)
    model = cp_model.CpModel()
    selected = [model.new_bool_var(f'select_{j}') for j in range(n)]
    choices = [[model.new_bool_var(f'mode_{j}_{m}') for m in range(modes)] for j in range(n)]
    end = [model.new_int_var(t0, end_shift, f'end_{j}') for j in range(n)]
    arcs, edge = [], {}
    empty = model.new_bool_var('empty')
    arcs.append((0, 0, empty))
    for j in range(n):
        model.add(sum(choices[j]) == selected[j])
        model.add(end[j] == end_shift).only_enforce_if(selected[j].Not())
        arcs.append((j+1, j+1, selected[j].Not()))
    for i in range(n+1):
        for j in range(n+1):
            if i == j:
                continue
            edge[i, j] = model.new_bool_var(f'edge_{i}_{j}')
            arcs.append((i, j, edge[i, j]))
    model.add_circuit(arcs)
    model.add(sum(selected) == 0).only_enforce_if(empty)
    model.add(sum(selected) >= 1).only_enforce_if(empty.Not())
    objective = []
    alpha, beta, _ = weights
    for j, job in enumerate(jobs):
        due = tick(job['deadline'])
        on_time = model.new_bool_var(f'on_time_{j}')
        model.add(on_time <= selected[j])
        model.add(end[j] <= due).only_enforce_if(on_time)
        model.add(end[j] >= due+1).only_enforce_if([selected[j], on_time.Not()])
        objective.append(int(round(bonus*cost_scale))*on_time)
        late = model.new_int_var(0, end_shift-t0, f'late_{j}')
        model.add_max_equality(late, [0, end[j]-max(t0, due)])
        objective.append(-int(round(alpha*cost_scale/time_scale))*late)
        objective.append(-int(round(unfinished*cost_scale))*(1-selected[j]))
        for i in range(n+1):
            if i == j+1:
                continue
            previous_arm = start_arm if i == 0 else jobs[i-1]['tray']
            switched = int(previous_arm != job['tray'])
            scan = 0. if i == 0 else float(scan_seconds[previous_arm])
            scanning_travel = 0. if i == 0 else float(scan_travel[previous_arm])
            for m in range(modes):
                use = model.new_bool_var(f'use_{i}_{j}_{m}')
                model.add(use <= edge[i, j+1])
                model.add(use <= choices[j][m])
                model.add(use >= edge[i, j+1]+choices[j][m]-1)
                dt = max(1, int(math.ceil((float(duration[j, m, switched])+scan)*time_scale)))
                previous_end = t0 if i == 0 else end[i-1]
                model.add(end[j] >= previous_end+dt).only_enforce_if(use)
                cost = beta*(float(travel[j, m, switched])+scanning_travel)+float(loss[j, m, switched])
                objective.append(-int(round(cost*cost_scale))*use)
    model.maximize(sum(objective))
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = float(time_limit)
    solver.parameters.num_search_workers = 1
    solver.parameters.random_seed = 0
    started = time.perf_counter()
    status = solver.solve(model)
    result = {'status': solver.status_name(status), 'wall_seconds': time.perf_counter()-started,
              'objective': None, 'bound': None, 'gap': None, 'order': [], 'modes': []}
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return result
    result.update(objective=float(solver.objective_value)/cost_scale,
                  bound=float(solver.best_objective_bound)/cost_scale)
    result['gap'] = max(0., result['bound']-result['objective'])
    node = 0
    if not solver.boolean_value(empty):
        for _ in range(n):
            following = next(j for j in range(n+1) if j != node and solver.boolean_value(edge[node, j]))
            if following == 0:
                break
            j = following-1
            result['order'].append(j)
            result['modes'].append(next(m for m in range(modes) if solver.boolean_value(choices[j][m])))
            node = following
    return result


def rolling_cp_sat(obs, env):
    slots = [int(slot) for slot in valid_slots(env)]
    inst = env._inst
    jobs = [{'job': int(env._slots[s]), 'tray': s//4,
             'deadline': float(inst.deadline[env._slots[s]])} for s in slots]
    duration, travel, loss = [np.zeros((len(slots), env.num_modes, 2)) for _ in range(3)]
    for j, slot in enumerate(slots):
        job = env._slots[slot]
        for mode in range(env.num_modes):
            for switched in (0, 1):
                est = MODEL.predict(inst.weight[job], inst.fragility[job], slot, mode, switched)
                duration[j, mode, switched] = est['seconds']
                travel[j, mode, switched] = est['travel']
                # Drops are a subset of failures; charge the non-drop failure term once.
                loss[j, mode, switched] = (env._w[2]*est['drop_probability'] +
                    env.cfg.failure_penalty*max(0., est['failure_probability']-est['drop_probability']))
    tray_mask = sum(1 << t for t in (0, 1) if any(s//4 == t for s in slots))
    scan_seconds = [SCAN_MODEL.seconds(arm, tray_mask)+CELL.clearance_seconds for arm in (0, 1)]
    scan_travel = [float(SCAN_MODEL.groups[arm, tray_mask]['travel'].mean()) for arm in (0, 1)]
    result = solve_queue_plan(jobs, duration, travel, loss, scan_seconds, scan_travel,
        env._arm, env._t, env.cfg.shift_length, env._w, env.cfg.on_time_bonus,
        env.cfg.unfinished_penalty, ROLLING_CP_SECONDS)
    fallback = not result['order']
    env._stats['solver_calls'] += 1
    env._stats['solver_optimal'] += result['status'] == 'OPTIMAL'
    env._stats['solver_fallback'] += fallback
    env._stats['planning_seconds'] += result['wall_seconds']
    env._events.append({'status': 'rolling_cp_sat', 't': env._t,
                        'solver_status': result['status'],
                        **{k: v for k, v in result.items() if k not in ('status', 'order', 'modes')},
                        'fallback': fallback})
    if fallback:
        return TUNED_ATCSR(obs, env)
    return result['modes'][0]*8+slots[result['order'][0]]


def solve_clairvoyant_relaxation(arrival, deadline, minimum_duration, horizon,
                                 time_limit=5., time_scale=100):
    """Upper bound on on-time completions in an optimistic scheduling relaxation."""
    from ortools.sat.python import cp_model
    arrival, deadline, minimum_duration = map(np.asarray, (arrival, deadline, minimum_duration))
    if not (arrival.shape == deadline.shape == minimum_duration.shape):
        raise ValueError('Ceiling inputs must have equal shapes')
    if (minimum_duration <= 0).any() or not np.isfinite(minimum_duration).all():
        raise ValueError('Ceiling durations must be positive and finite')
    model = cp_model.CpModel()
    intervals, included = [], []
    h = int(math.ceil(horizon*time_scale))
    for j, (release, due, duration) in enumerate(zip(arrival, deadline, minimum_duration)):
        take = model.new_bool_var(f'take_{j}')
        included.append(take)
        # Round in the optimistic direction so time discretisation preserves the bound.
        start = model.new_int_var(0, h, f'start_{j}')
        end = model.new_int_var(0, h, f'end_{j}')
        p = max(0, int(math.floor(float(duration)*time_scale)))
        intervals.append(model.new_optional_interval_var(start, p, end, take, f'job_{j}'))
        model.add(start >= int(math.floor(float(release)*time_scale))).only_enforce_if(take)
        model.add(end <= min(h, int(math.ceil(float(due)*time_scale)))).only_enforce_if(take)
    model.add_no_overlap(intervals)
    model.maximize(sum(included))
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = float(time_limit)
    solver.parameters.num_search_workers = 1
    solver.parameters.random_seed = 0
    status = solver.solve(model)
    feasible = status in (cp_model.OPTIMAL, cp_model.FEASIBLE)
    # UNKNOWN has no incumbent. The job count is still an independently valid bound.
    upper = min(len(arrival), int(math.ceil(solver.best_objective_bound))) if feasible else len(arrival)
    return {'status': solver.status_name(status),
            'on_time_incumbent': int(round(solver.objective_value)) if feasible else None,
            'on_time_upper_bound': upper,
            'gap': max(0, upper-int(round(solver.objective_value))) if feasible else None}


def clairvoyant_ceiling(instance, cfg=ENV):
    minimum = []
    for weight, fragility, tray in zip(instance.weight, instance.fragility, instance.tray):
        # EventBackend samples one of these rows. Every realised pick is at least this long.
        minimum.append(min(float(MODEL.neighbors(weight, fragility, slot, mode, switched)['elapsed'].min())
            for slot in range(int(tray)*4, int(tray)*4+4)
            for mode in range(2 if cfg.careful_mode else 1) for switched in (0, 1)))
    result = solve_clairvoyant_relaxation(instance.arrival, instance.deadline,
        minimum, cfg.shift_length, CLAIRVOYANT_CP_SECONDS)
    result.update(surrogate_score_upper_bound=cfg.on_time_bonus*result['on_time_upper_bound'],
                  physics_score_upper_bound=cfg.on_time_bonus*len(instance),
                  scope='calibrated_event_relaxation', reference='Clairvoyant CP-SAT')
    return result


def check_solver_known_answers():
    # Exhaustive independent solution of a small maximum-on-time problem.
    from itertools import permutations
    arrival, deadline, duration = [0., 0., 1.], [2., 3., 4.], [2., 1., 2.]
    best = 0
    for length in range(4):
        for order in permutations(range(3), length):
            t, valid = 0., True
            for j in order:
                t = max(t, arrival[j])+duration[j]
                valid &= t <= min(4., deadline[j])
            if valid:
                best = max(best, length)
    result = solve_clairvoyant_relaxation(arrival, deadline, duration, 4., 2.)
    if result['status'] != 'OPTIMAL' or result['on_time_incumbent'] != best or result['on_time_upper_bound'] != best:
        raise RuntimeError(f'Clairvoyant known-answer check failed: {result}, expected {best}')
    jobs = [{'tray': 0, 'deadline': 1.}, {'tray': 1, 'deadline': 3.}]
    d = np.ones((2, 2, 2)); d[:, :, 1] = 2.; d[:, 1, :] += .5
    # Full enumeration of order, subset and mode for the rolling objective.
    best_score = -np.inf
    for length in range(3):
        for order in permutations(range(2), length):
            for modes in __import__('itertools').product(range(2), repeat=length):
                t, arm, score = 0., 0, -float(2-length)
                for j, mode in zip(order, modes):
                    t += d[j, mode, int(arm != jobs[j]['tray'])]
                    arm = jobs[j]['tray']
                    if t > 4.:
                        break
                    score += float(t <= jobs[j]['deadline'])-.1*max(0., t-jobs[j]['deadline'])
                else:
                    score -= .1*sum(max(0., 4.-jobs[j]['deadline']) for j in set(range(2))-set(order))
                    best_score = max(best_score, score)
    result = solve_queue_plan(jobs, d, np.zeros_like(d), np.zeros_like(d), [0., 0.],
        [0., 0.], 0, 0., 4., [.1, 0., 0.], time_limit=2.)
    if result['status'] != 'OPTIMAL' or abs(result['objective']-best_score) > 1e-6:
        raise RuntimeError(f'Rolling known-answer check failed: {result}, expected {best_score}')
    return {'clairvoyant_optimum': best, 'rolling_optimum': best_score}

TUNED_ATCSR = atcsr_policy()
ATCSR_PARAMETERS = {'k1': 1., 'k2': 1.}


In [ ]:
solver_check = check_solver_known_answers()
print('Known-answer solver checks:', solver_check)

TUNED_ATCSR = atcsr_policy()
ATCSR_PARAMETERS = {'k1': 1., 'k2': 1.}
validation_instances = make_instances(ENV, FAMILIAR, N_VAL_SHIFTS, VAL_SEED)
grid = [(k1, k2) for k1 in (.5, 1., 2.) for k2 in (.5, 1., 2.)]
tuning_key = canonical_hash({'experiment': EXPERIMENT_KEY, 'grid': grid,
    'seed': VAL_SEED, 'count': N_VAL_SHIFTS, 'weights': EVAL_WEIGHTS})
tuning_path = RESULTS_DIR/'atcsr_validation.json'
if tuning_path.exists():
    tuning = json.loads(tuning_path.read_text())
    if tuning['key'] != tuning_key:
        raise RuntimeError('ATCSR validation cache differs; use a fresh RUN_DIR')
else:
    trials = []
    for k1, k2 in grid:
        score = float(np.mean([row['score'] for row in evaluate(
            atcsr_policy(k1, k2), validation_instances)]))
        trials.append({'k1': k1, 'k2': k2, 'validation_score': score})
    best = max(trials, key=lambda row: (row['validation_score'], -row['k1'], -row['k2']))
    tuning = {'key': tuning_key, 'parameters': {name: best[name] for name in ('k1', 'k2')},
              'trials': trials, 'source': 'validation_only'}
    atomic_json(tuning_path, tuning)
ATCSR_PARAMETERS = tuning['parameters']
TUNED_ATCSR = atcsr_policy(**ATCSR_PARAMETERS)
display(pd.DataFrame(tuning['trials']).sort_values('validation_score', ascending=False))


## 10. Learning models: masked PPO and masked double dueling DQN

Apply the same scoring network to each pocket and use masks to select valid actions.
Both PPO and DQN use 79-dimensional observations by default and 16 actions. The optional vision profile
uses 95 dimensions. Each training seed is validated on shared shifts. DQN's terminal target sets future Q to zero and allows an all-false mask at termination.


In [ ]:
@dataclass(frozen=True)
class ObsLayout:
    num_slots: int
    slot_dim: int
    global_dim: int
    num_modes: int

    @property
    def num_actions(self):
        return self.num_slots * self.num_modes


def layout_of(env):
    return ObsLayout(env.cfg.num_slots, env.slot_dim, env.global_dim, env.num_modes)


def mlp(sizes):
    # Linear layers with ReLU in between (none after the last one)
    layers = []
    for i, (a, b) in enumerate(zip(sizes, sizes[1:])):
        layers.append(nn.Linear(a, b))
        if i < len(sizes) - 2:
            layers.append(nn.ReLU())
    return nn.Sequential(*layers)


class QueueEncoder(nn.Module):
    def __init__(self, layout, hidden=64):
        super().__init__()
        self.layout = layout
        self.job = nn.Sequential(mlp([layout.slot_dim + layout.global_dim, hidden, hidden]), nn.ReLU())
        self.context_dim = 2 * hidden

    def forward(self, obs):
        # returns job embeddings (batch, K, hidden), queue context (batch, 2 * hidden), global features
        k, s = self.layout.num_slots, self.layout.slot_dim
        slots = obs[:, : k * s].reshape(-1, k, s)
        glob = obs[:, k * s :]
        present = slots[..., 0] > 0.5
        h = self.job(torch.cat([slots, glob.unsqueeze(1).expand(-1, k, -1)], dim=-1))
        mask = present.unsqueeze(-1)
        mean = (h * mask).sum(1) / mask.sum(1).clamp(min=1)
        mx = h.masked_fill(~mask, float('-inf')).max(1).values
        mx = torch.where(present.any(1, keepdim=True), mx, torch.zeros_like(mx))  # empty queue: zeros
        return h, torch.cat([mean, mx], dim=-1), glob


class JobScorer(nn.Module):
    # One score per (job slot, pick mode), laid out as action = mode * K + slot
    def __init__(self, layout, embed_dim, context_dim, hidden=64):
        super().__init__()
        self.layout = layout
        self.net = mlp([embed_dim + context_dim, hidden, layout.num_modes])

    def forward(self, h, context):
        k = self.layout.num_slots
        scores = self.net(torch.cat([h, context.unsqueeze(1).expand(-1, k, -1)], dim=-1))  # (batch, K, modes)
        return scores.transpose(1, 2).reshape(h.shape[0], -1)


class ValueHead(nn.Module):
    def __init__(self, context_dim, global_dim, hidden=64):
        super().__init__()
        self.net = mlp([context_dim + global_dim, hidden, 1])

    def forward(self, context, glob):
        return self.net(torch.cat([context, glob], dim=-1))


In [ ]:
PPO_PARAMS = dict(
    learning_rate=3e-4,
    n_steps=256,      # rollout length per env; 8 envs give 2048 transitions per update
    batch_size=256,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.01,    # entropy bonus, the exploration knob (DQN uses epsilon instead)
)


class JobScoringExtractor(nn.Module):
    # Fills SB3's mlp_extractor slot: latent_pi is already the action logits, latent_vf a queue embedding
    def __init__(self, layout, hidden):
        super().__init__()
        self.encoder = QueueEncoder(layout, hidden)
        self.scorer = JobScorer(layout, hidden, self.encoder.context_dim, hidden)
        self.value = nn.Sequential(mlp([self.encoder.context_dim + layout.global_dim, hidden, hidden]), nn.ReLU())
        self.latent_dim_pi = layout.num_actions
        self.latent_dim_vf = hidden

    def forward(self, features):
        h, context, glob = self.encoder(features)
        return self.scorer(h, context), self.value(torch.cat([context, glob], dim=-1))

    def forward_actor(self, features):
        h, context, _ = self.encoder(features)
        return self.scorer(h, context)

    def forward_critic(self, features):
        _, context, glob = self.encoder(features)
        return self.value(torch.cat([context, glob], dim=-1))


class JobScoringPolicy(MaskableActorCriticPolicy):
    def __init__(self, *args, layout, hidden=64, **kwargs):
        self.layout, self.hidden = layout, hidden
        super().__init__(*args, **kwargs)

    def _build_mlp_extractor(self):
        self.mlp_extractor = JobScoringExtractor(self.layout, self.hidden)

    def _build(self, lr_schedule):
        super()._build(lr_schedule)
        # The scorer already outputs logits: drop SB3's extra action layer and start the logits small
        self.action_net = nn.Identity()
        nn.init.orthogonal_(self.mlp_extractor.scorer.net[-1].weight, gain=0.01)
        nn.init.zeros_(self.mlp_extractor.scorer.net[-1].bias)
        self.optimizer = self.optimizer_class(self.parameters(), lr=lr_schedule(1), **self.optimizer_kwargs)

    def _get_constructor_parameters(self):
        return super()._get_constructor_parameters() | {'layout': self.layout, 'hidden': self.hidden}


def ppo_policy(model):
    # Deterministic masked policy for evaluation
    def policy(obs, env):
        action, _ = model.predict(obs, action_masks=env.action_masks(), deterministic=True)
        return int(action)

    return policy


class ValidationCallback(BaseCallback):
    # Runs the validator every VAL_EVERY steps and at the end of training
    def __init__(self, validator):
        super().__init__()
        self.validator, self.next_eval = validator, 0

    def _on_step(self):
        if self.num_timesteps >= self.next_eval:
            self.next_eval += VAL_EVERY
            self._validate()
        return True

    def _on_training_end(self):
        self._validate()

    def _validate(self):
        save = lambda: self.model.save(self.validator.out_dir / 'best_model')
        self.validator(ppo_policy(self.model), self.num_timesteps, save)




def load_ppo(path):
    # Evaluation runs one shift at a time, which is faster on the CPU
    return ppo_policy(MaskablePPO.load(path, device='cpu'))


In [ ]:
@dataclass
class DQNConfig:
    timesteps: int = DQN_TIMESTEPS
    n_envs: int = 8
    lr: float = 1e-4
    gamma: float = 0.99
    batch_size: int = 128
    buffer_size: int = 100_000
    learning_starts: int = 5_000
    train_freq: int = 4           # environment steps per gradient update
    tau: float = 0.005            # soft update rate of the target network
    eps_start: float = 1.0
    eps_end: float = 0.05
    eps_fraction: float = 0.3     # share of training over which epsilon decays
    warmup_edf_steps: int = 0     # optionally act with EDF first to seed the replay buffer
    hidden: int = 64
    grad_clip: float = 10.0
    device: str = DEVICE


class DuelingJobQNet(nn.Module):
    def __init__(self, layout, hidden=64):
        super().__init__()
        self.encoder = QueueEncoder(layout, hidden)
        self.advantage = JobScorer(layout, hidden, self.encoder.context_dim, hidden)
        self.value = ValueHead(self.encoder.context_dim, layout.global_dim, hidden)

    def forward(self, obs, mask):
        h, context, glob = self.encoder(obs)
        adv = self.advantage(h, context)
        adv_mean = (adv * mask).sum(1, keepdim=True) / mask.sum(1, keepdim=True).clamp(min=1)
        q = self.value(context, glob) + adv - adv_mean
        return q.masked_fill(~mask, -1e9)


class ReplayBuffer:
    def __init__(self, capacity, obs_dim, num_actions):
        self.capacity, self.size, self.pos = capacity, 0, 0
        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.mask = np.zeros((capacity, num_actions), dtype=bool)
        self.next_mask = np.zeros((capacity, num_actions), dtype=bool)
        self.action = np.zeros(capacity, dtype=np.int64)
        self.reward = np.zeros(capacity, dtype=np.float32)
        self.done = np.zeros(capacity, dtype=np.float32)

    def add(self, obs, mask, action, reward, next_obs, next_mask, done):
        i = self.pos
        self.obs[i], self.mask[i], self.action[i], self.reward[i] = obs, mask, action, reward
        self.next_obs[i], self.next_mask[i], self.done[i] = next_obs, next_mask, done
        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, rng, batch_size, device):
        idx = rng.integers(self.size, size=batch_size)
        names = ('obs', 'mask', 'action', 'reward', 'next_obs', 'next_mask', 'done')
        return {name: torch.as_tensor(getattr(self, name)[idx], device=device) for name in names}


def dqn_policy(net):
    # Greedy masked policy for evaluation
    device = next(net.parameters()).device

    def policy(obs, env):
        with torch.no_grad():
            q = net(torch.as_tensor(obs[None], device=device), torch.as_tensor(env.action_masks()[None], device=device))
        return int(q.argmax(1))

    return policy


def dqn_update(online, target, optimizer, batch, cfg):
    q = online(batch['obs'], batch['mask']).gather(1, batch['action'][:, None]).squeeze(1)
    with torch.no_grad():
        best_next = online(batch['next_obs'], batch['next_mask']).argmax(1, keepdim=True)  # online net picks
        next_q = target(batch['next_obs'], batch['next_mask']).gather(1, best_next).squeeze(1)  # target net scores
        done = batch['done'].bool()
        next_q = torch.where(done, torch.zeros_like(next_q), next_q)  # no future after the shift ends
        y = batch['reward'] + cfg.gamma * next_q
    loss = F.smooth_l1_loss(q, y)
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(online.parameters(), cfg.grad_clip)
    optimizer.step()
    with torch.no_grad():
        for p, tp in zip(online.parameters(), target.parameters()):
            tp.lerp_(p, cfg.tau)




def load_dqn(path, hidden=64):
    net = DuelingJobQNet(layout_of(LVHMSchedulingEnv(ENV)), hidden)
    net.load_state_dict(torch.load(path, map_location='cpu', weights_only=True))
    return dqn_policy(net.eval())


In [ ]:
def atomic_torch(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix+'.tmp')
    torch.save(value, temporary)
    temporary.replace(path)


def capture_random_state():
    return {'numpy': np.random.get_state(), 'python': random.getstate(),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def restore_random_state(state):
    np.random.set_state(state['numpy'])
    random.setstate(state['python'])
    torch.set_rng_state(state['torch'].cpu())
    if state['cuda'] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([value.cpu() for value in state['cuda']])


def capture_environment(env):
    if not isinstance(env.backend, EventBackend):
        raise ValueError('Training checkpoints support the calibrated event backend only')
    excluded = {'backend', 'cfg', 'action_space', 'observation_space'}
    backend_excluded = {'model', 'scan_model'}
    return {'env': copy.deepcopy({key: value for key, value in vars(env).items() if key not in excluded}),
            'backend': copy.deepcopy({key: value for key, value in vars(env.backend).items()
                                      if key not in backend_excluded})}


def restore_environment(env, state):
    if not isinstance(env.backend, EventBackend):
        raise ValueError('Cannot restore a training checkpoint into a physics worker')
    vars(env).update(copy.deepcopy(state['env']))
    vars(env.backend).update(copy.deepcopy(state['backend']))


def training_resume_key(algorithm, seed, parameters, timesteps):
    return canonical_hash({'experiment': EXPERIMENT_KEY, 'algorithm': algorithm,
        'seed': seed, 'parameters': parameters, 'timesteps': timesteps,
        'val_seed': VAL_SEED, 'val_shifts': N_VAL_SHIFTS, 'val_every': VAL_EVERY})


class TrainingPause(Exception):
    pass


class CheckpointMaskablePPO(MaskablePPO):
    def _excluded_save_params(self):
        return super()._excluded_save_params()+['_resume_hook']

    def train(self):
        super().train()
        hook = getattr(self, '_resume_hook', None)
        if hook is not None:
            hook(self)


def save_ppo_resume(model, validator, out_dir, key):
    # Publish a pointer only after both members of a generation have been saved.
    step = model.num_timesteps
    stem = f'resume_{step:09d}'
    model_path, state_path = out_dir/f'{stem}.zip', out_dir/f'{stem}.pt'
    temporary = out_dir/f'{stem}.tmp.zip'
    model.save(temporary)
    temporary.replace(model_path)
    vector = model.get_env()
    state = {'key': key, 'step': step, 'random': capture_random_state(),
        'envs': [capture_environment(env.unwrapped) for env in vector.envs],
        'monitors': [{name: copy.deepcopy(getattr(env, name)) for name in
            ('needs_reset', 'rewards', 'total_steps', 'current_reset_info',
             'episode_returns', 'episode_lengths', 'episode_times', 't_start')}
            for env in vector.envs],
        'vector': {name: copy.deepcopy(getattr(vector, name))
                   for name in ('buf_obs', 'buf_dones', 'buf_rews', 'buf_infos')},
        'last_obs': copy.deepcopy(model._last_obs),
        'last_episode_starts': copy.deepcopy(model._last_episode_starts)}
    atomic_torch(state_path, state)
    atomic_json(out_dir/'resume.json', {'key': key, 'step': step,
        'model': model_path.name, 'state': state_path.name})
    # A failed publication leaves the previous pointer and generation usable.
    for old in out_dir.glob('resume_*'):
        if old not in (model_path, state_path):
            old.unlink()


def restore_ppo_resume(model, vector, state):
    if len(vector.envs) != len(state['envs']):
        raise RuntimeError('PPO checkpoint environment count differs')
    for env, saved in zip(vector.envs, state['envs']):
        restore_environment(env.unwrapped, saved)
    for env, saved in zip(vector.envs, state['monitors']):
        for name, value in saved.items():
            setattr(env, name, copy.deepcopy(value))
    for name, value in state['vector'].items():
        setattr(vector, name, copy.deepcopy(value))
    model._last_obs = copy.deepcopy(state['last_obs'])
    model._last_episode_starts = copy.deepcopy(state['last_episode_starts'])
    if model.num_timesteps != state['step']:
        raise RuntimeError('PPO checkpoint model/state generations differ')
    restore_random_state(state['random'])


def train_ppo(seed, timesteps, out_dir, device=DEVICE, n_envs=8, hidden=64, stop_after=None):
    out_dir.mkdir(parents=True, exist_ok=True)
    parameters = dict(PPO_PARAMS, n_envs=n_envs, hidden=hidden, device=device)
    key = training_resume_key('ppo', seed, parameters, timesteps)
    vector = make_vec_env(LVHMSchedulingEnv, n_envs, seed=seed,
        env_kwargs={'config': ENV}, vec_env_cls=DummyVecEnv)
    probe = LVHMSchedulingEnv(ENV)
    layout = layout_of(probe)
    probe.close()
    pointer_path = out_dir/'resume.json'
    validator = Validator(out_dir)
    resumed_from = 0
    try:
        if pointer_path.exists():
            pointer = json.loads(pointer_path.read_text())
            if pointer['key'] != key:
                raise RuntimeError('PPO resume configuration differs; use a fresh RUN_DIR')
            state = torch.load(out_dir/pointer['state'], map_location='cpu', weights_only=False)
            if state['key'] != key:
                raise RuntimeError('PPO resume state provenance differs')
            model = CheckpointMaskablePPO.load(out_dir/pointer['model'], env=vector,
                device=device, force_reset=False)
            restore_ppo_resume(model, vector, state)
            resumed_from = model.num_timesteps
        else:
            model = CheckpointMaskablePPO(JobScoringPolicy, vector,
                policy_kwargs={'layout': layout, 'hidden': hidden}, seed=seed,
                device=device, **PPO_PARAMS)
        last_saved = resumed_from
        def checkpoint(current):
            nonlocal last_saved
            if (current.num_timesteps-last_saved >= CHECKPOINT_EVERY
                    or current.num_timesteps >= timesteps
                    or (stop_after is not None and current.num_timesteps >= stop_after)):
                save_ppo_resume(current, validator, out_dir, key)
                last_saved = current.num_timesteps
            if stop_after is not None and current.num_timesteps >= stop_after:
                raise TrainingPause()
        model._resume_hook = checkpoint
        started = time.perf_counter()
        callback = ValidationCallback(validator)
        callback.next_eval = ((resumed_from//VAL_EVERY)+1)*VAL_EVERY if resumed_from else 0
        if resumed_from < timesteps:
            try:
                model.learn(timesteps-resumed_from, callback=callback,
                    reset_num_timesteps=(resumed_from == 0))
            except TrainingPause:
                pass
        save_ppo_resume(model, validator, out_dir, key)
        return {'seed': seed, 'device': str(model.device),
            'minutes': round((time.perf_counter()-started)/60, 1),
            'best_validation_score': validator.best, 'steps': model.num_timesteps,
            'resumed_from': resumed_from, 'complete': model.num_timesteps >= timesteps}
    finally:
        vector.close()


def replay_state(buffer):
    names = ('obs', 'next_obs', 'mask', 'next_mask', 'action', 'reward', 'done')
    return {'capacity': buffer.capacity, 'size': buffer.size, 'pos': buffer.pos,
            'arrays': {name: getattr(buffer, name).copy() for name in names}}


def restore_replay(buffer, state):
    if buffer.capacity != state['capacity']:
        raise RuntimeError('DQN replay capacity differs')
    for name, values in state['arrays'].items():
        if getattr(buffer, name).shape != values.shape:
            raise RuntimeError(f'DQN replay shape differs: {name}')
        getattr(buffer, name)[:] = values
    buffer.size, buffer.pos = state['size'], state['pos']


def train_dqn(seed, cfg, out_dir, stop_after=None):
    out_dir.mkdir(parents=True, exist_ok=True)
    key = training_resume_key('dqn', seed, asdict(cfg), cfg.timesteps)
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    device = torch.device(cfg.device)
    envs = [LVHMSchedulingEnv(ENV) for _ in range(cfg.n_envs)]
    layout = layout_of(envs[0])
    online = DuelingJobQNet(layout, cfg.hidden).to(device)
    target = DuelingJobQNet(layout, cfg.hidden).to(device)
    target.load_state_dict(online.state_dict())
    target.requires_grad_(False)
    optimizer = torch.optim.Adam(online.parameters(), lr=cfg.lr)
    buffer = ReplayBuffer(cfg.buffer_size, envs[0].observation_space.shape[0], layout.num_actions)
    validator = Validator(out_dir)
    def validate(step):
        validator(dqn_policy(online), step,
            lambda: atomic_torch(out_dir/'best_model.pt', online.state_dict()))
    path = out_dir/'resume_dqn.pt'
    step, next_val, updates_due, updates, resumed_from = 0, 0, 0., 0, 0
    if path.exists():
        state = torch.load(path, map_location='cpu', weights_only=False)
        if state['key'] != key:
            raise RuntimeError('DQN resume configuration differs; use a fresh RUN_DIR')
        online.load_state_dict(state['online'])
        target.load_state_dict(state['target'])
        optimizer.load_state_dict(state['optimizer'])
        restore_replay(buffer, state['replay'])
        for env, saved in zip(envs, state['envs']):
            restore_environment(env, saved)
        obs, masks = state['obs'], state['masks']
        step, next_val, updates_due, updates = (state[name] for name in
            ('step', 'next_val', 'updates_due', 'updates'))
        rng.bit_generator.state = state['rng']
        restore_random_state(state['random'])
        resumed_from = step
    else:
        obs = np.stack([env.reset(seed=seed*1000+i)[0] for i, env in enumerate(envs)])
        masks = np.stack([env.action_masks() for env in envs])
    def save():
        atomic_torch(path, {'key': key, 'step': step, 'next_val': next_val,
            'updates_due': updates_due, 'updates': updates,
            'online': online.state_dict(), 'target': target.state_dict(),
            'optimizer': optimizer.state_dict(), 'replay': replay_state(buffer),
            'obs': obs.copy(), 'masks': masks.copy(),
            'envs': [capture_environment(env) for env in envs],
            'rng': rng.bit_generator.state, 'random': capture_random_state()})
    last_saved = step
    started = time.perf_counter()
    try:
        while step < cfg.timesteps:
            if step >= next_val:
                validate(step)
                next_val += VAL_EVERY
            eps = max(cfg.eps_end, cfg.eps_start-(cfg.eps_start-cfg.eps_end)*step/(cfg.eps_fraction*cfg.timesteps))
            with torch.no_grad():
                greedy = online(torch.as_tensor(obs, device=device),
                    torch.as_tensor(masks, device=device)).argmax(1).cpu().numpy()
            for i, env in enumerate(envs):
                if step < cfg.warmup_edf_steps:
                    action = edf(obs[i], env)
                elif rng.random() < eps:
                    action = int(rng.choice(np.flatnonzero(masks[i])))
                else:
                    action = int(greedy[i])
                next_obs, reward, terminated, truncated, _ = env.step(action)
                next_mask = env.action_masks()
                buffer.add(obs[i], masks[i], action, reward, next_obs, next_mask, float(terminated or truncated))
                if terminated or truncated:
                    next_obs, _ = env.reset()
                    next_mask = env.action_masks()
                obs[i], masks[i] = next_obs, next_mask
            step += cfg.n_envs
            if step >= cfg.learning_starts:
                updates_due += cfg.n_envs/cfg.train_freq
                while updates_due >= 1:
                    updates_due -= 1
                    dqn_update(online, target, optimizer,
                        buffer.sample(rng, cfg.batch_size, device), cfg)
                    updates += 1
            if step-last_saved >= CHECKPOINT_EVERY:
                save()
                last_saved = step
            if stop_after is not None and step >= stop_after:
                save()
                break
        if step >= cfg.timesteps:
            validate(step)
        save()
        if step >= cfg.timesteps and updates == 0:
            raise RuntimeError('DQN completed without a learning update')
        return {'seed': seed, 'device': str(device),
            'minutes': round((time.perf_counter()-started)/60, 1),
            'best_validation_score': validator.best, 'steps': step,
            'resumed_from': resumed_from, 'updates': updates, 'complete': step >= cfg.timesteps}
    finally:
        for env in envs:
            env.close()


def check_training_resume():
    # Isolated directories; no production checkpoints or RNG state are changed.
    saved_random = capture_random_state()
    saved_threads = torch.get_num_threads()
    try:
        torch.set_num_threads(1)
        with tempfile.TemporaryDirectory(prefix='vision-training-smoke-') as directory:
            directory = Path(directory)
            ppo = directory/'ppo'
            first = train_ppo(731, 1024, ppo, device='cpu', n_envs=2, hidden=16, stop_after=512)
            second = train_ppo(731, 1024, ppo, device='cpu', n_envs=2, hidden=16)
            dqn = directory/'dqn'
            cfg = DQNConfig(timesteps=96, n_envs=2, hidden=16, device='cpu',
                learning_starts=8, batch_size=8, buffer_size=128)
            first_dqn = train_dqn(732, cfg, dqn, stop_after=48)
            second_dqn = train_dqn(732, cfg, dqn)
            if first['complete'] or second['resumed_from'] != 512 or not second['complete']:
                raise RuntimeError('PPO save/load/continue smoke check failed')
            if (first_dqn['complete'] or second_dqn['resumed_from'] != 48
                    or second_dqn['updates'] <= first_dqn['updates'] or not second_dqn['complete']):
                raise RuntimeError('DQN save/load/continue smoke check failed')
            return {'ppo_resumed_from': second['resumed_from'],
                    'dqn_resumed_from': second_dqn['resumed_from'], 'dqn_updates': second_dqn['updates']}
    finally:
        restore_random_state(saved_random)
        torch.set_num_threads(saved_threads)


## 11. Environment contract, training resume, and preflight checks

Full mode runs 10 seeds × 300,000 steps for each of PPO and DQN, alternating the algorithms for each seed.
Quick mode is a short run with one seed and is not reported as a research result.

Before training, check the environment contract, known solver answers, and short PPO/DQN save/load/continue runs.
PPO saves after a rollout update; DQN saves after a full vector step and its updates.
Restore the optimizer, step, environment, random states, and DQN replay buffer.
PPO checkpoints update the atomic pointer only after both model and state files are complete.

Resume or reuse requires matching experiment version, observation profile, settings, measurements, and training settings.
Use a new version and output path after changing code or measurement conditions. Reusing a completed model and
resuming interrupted training are handled separately. `RETRAIN=True` clears results in that training path.


In [ ]:
from stable_baselines3.common.env_checker import check_env
class MaskedChecker(gym.ActionWrapper):
    def action(self, action):
        mask = self.env.action_masks()
        return int(action) if mask[action] else int(np.flatnonzero(mask)[0])

probe = MaskedChecker(LVHMSchedulingEnv(ENV))
try:
    check_env(probe, warn=True)
finally:
    probe.close()
sanity_instances = make_instances(ENV, FAMILIAR, 3, 40_000)
sanity = {name: mean_metrics(evaluate(policy, sanity_instances))
          for name, policy in rule_baselines().items()}
display(pd.DataFrame(sanity).T.round(2))
if RUN_TRAINING_SMOKE:
    print('Training save/load/continue smoke:', check_training_resume())

def checkpoint_key(algorithm, seed):
    parameters = PPO_PARAMS if algorithm == 'ppo' else asdict(DQNConfig())
    return canonical_hash({'experiment': EXPERIMENT_KEY,
        'algorithm': algorithm, 'parameters': parameters, 'seed': seed,
        'timesteps': PPO_TIMESTEPS if algorithm == 'ppo' else DQN_TIMESTEPS,
        'val_seed': VAL_SEED, 'val_shifts': N_VAL_SHIFTS, 'val_every': VAL_EVERY})

def train_or_reuse(algorithm, seed):
    out_dir = RUN_DIR/algorithm/f'seed{seed}'
    out_dir.mkdir(parents=True, exist_ok=True)
    key = checkpoint_key(algorithm, seed)
    metadata_path = out_dir/'training_meta.json'
    suffix = '.zip' if algorithm == 'ppo' else '.pt'
    if metadata_path.exists() and not RETRAIN:
        meta = json.loads(metadata_path.read_text())
        if meta['key'] != key:
            raise RuntimeError(f'Checkpoint provenance differs: {out_dir}. Use a fresh RUN_DIR.')
        if meta['training']['complete'] and (out_dir/f'best_model{suffix}').exists():
            return meta['training']
    if RETRAIN:
        for pattern in ('candidate_*', 'best_model.*', 'resume*', 'candidates.json', 'validation.csv', 'training_meta.json'):
            for previous in out_dir.glob(pattern):
                previous.unlink()
    if algorithm == 'ppo':
        result = train_ppo(seed, PPO_TIMESTEPS, out_dir)
    else:
        cfg = DQNConfig(timesteps=DQN_TIMESTEPS, **DQN_TRAINING_OVERRIDES)
        result = train_dqn(seed, cfg, out_dir)
    atomic_json(metadata_path, {'key': key, 'experiment_version': EXPERIMENT_VERSION,
        'experiment_key': EXPERIMENT_KEY, 'training': result})
    if not result['complete']:
        raise RuntimeError(f'Training incomplete: {algorithm} seed {seed}')
    return result


In [ ]:
training_rows = []
for seed in SEEDS:
    for algorithm in ('ppo', 'dqn'):
        print(f'Training {algorithm.upper()} seed {seed}')
        training_rows.append({'algorithm': algorithm, **train_or_reuse(algorithm, seed)})
training_df = pd.DataFrame(training_rows)
training_df.to_csv(RESULTS_DIR/'training.csv', index=False)
display(training_df)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, algorithm in zip(axes, ('ppo', 'dqn')):
    for seed in SEEDS:
        log = pd.read_csv(RUN_DIR/algorithm/f'seed{seed}'/'validation.csv')
        ax.plot(log['timesteps'], log['score'], marker='.', label=f'seed {seed}')
    ax.set(title=f'{algorithm.upper()}: surrogate validation', xlabel='Environment transitions', ylabel='Score')
    ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR/'learning_curves.png', dpi=160)
plt.show()


## 12. Final checkpoint selection with physical validation

Evaluate up to three candidates retained by surrogate validation on the same familiar validation shifts with RGB+physics.
Select the candidate with the highest mean score. Do not change candidates, mode rules, or the detector after viewing
test results. Completed validation trials are reused when the experiment version, settings, measurements,
and evaluation conditions match.


In [ ]:
def load_cached_shifts(log_path):
    cached = {}
    if log_path.exists():
        lines = log_path.read_text().splitlines()
        for index, line in enumerate(lines):
            try:
                entry = json.loads(line)
            except json.JSONDecodeError:
                if index != len(lines)-1:
                    raise
                # An interrupted write at the end is dropped before resuming.
                break
            cached[entry['shift']] = entry
        log_path.write_text(''.join(json.dumps(cached[k])+'\n' for k in sorted(cached)), encoding='utf-8')
    return cached

def cached_trials(policy, instances, cfg, weights, backend_factory, directory, signature):
    directory.mkdir(parents=True, exist_ok=True)
    key = canonical_hash({'experiment': EXPERIMENT_KEY,
                          'signature': signature, 'cfg': asdict(cfg), 'weights': weights,
                          'instances': [{'scene_seed': i.scene_seed, 'appearance': i.appearance,
                            'arrival': i.arrival.tolist(), 'deadline': i.deadline.tolist(),
                            'weight': i.weight.tolist(), 'fragility': i.fragility.tolist(),
                            'tray': i.tray.tolist()} for i in instances]})
    metadata_path = directory/'meta.json'
    if metadata_path.exists() and json.loads(metadata_path.read_text())['key'] != key:
        raise RuntimeError(f'Evaluation cache changed: {directory}. Use a fresh RUN_DIR.')
    atomic_json(metadata_path, {'key': key, 'experiment_version': EXPERIMENT_VERSION,
        'experiment_key': EXPERIMENT_KEY, 'signature': signature})
    log_path = directory/'shifts.jsonl'
    cached = load_cached_shifts(log_path)
    env = LVHMSchedulingEnv(cfg, backend_factory())
    started = time.perf_counter()
    try:
        with log_path.open('a', encoding='utf-8') as stream:
            for shift, instance in enumerate(instances):
                if shift in cached:
                    continue
                metrics = run_episode(env, policy, instance, weights)
                entry = {'shift': shift, 'scene_seed': instance.scene_seed, **metrics}
                # Event traces make timing/failure semantics inspectable without storing every frame.
                atomic_json(directory/f'events_{shift:03d}.json', env._events)
                stream.write(json.dumps(entry, allow_nan=False)+'\n')
                stream.flush()
                cached[shift] = entry
                if (shift+1) % 5 == 0 or shift+1 == len(instances):
                    print(f'{directory.name}: {shift+1}/{len(instances)} shifts, {(time.perf_counter()-started)/60:.1f} min')
    finally:
        env.close()
    return [cached[i] for i in range(len(instances))]

def load_algorithm(algorithm, path):
    return load_ppo(path) if algorithm == 'ppo' else load_dqn(path)

selected = {}
physical_validation_rows = []
val_instances = make_instances(ENV, FAMILIAR, N_VAL_SHIFTS, VAL_SEED)
for algorithm in ('ppo', 'dqn'):
    selected[algorithm] = []
    for seed in SEEDS:
        out_dir = RUN_DIR/algorithm/f'seed{seed}'
        candidates = json.loads((out_dir/'candidates.json').read_text())
        scores = []
        for candidate in candidates:
            path = out_dir/candidate['file']
            signature = {'algorithm': algorithm, 'seed': seed, 'checkpoint_hash': file_hash(path),
                         'backend': 'physics_camera', 'stage': 'validation'}
            result = cached_trials(load_algorithm(algorithm, path), val_instances, ENV, EVAL_WEIGHTS,
                lambda: PhysicsBackend(WORKER), out_dir/'physics_validation'/path.stem, signature)
            score = float(np.mean([r['score'] for r in result]))
            scores.append((score, candidate['timesteps'], path))
            physical_validation_rows.append({'algorithm': algorithm, 'seed': seed,
                'timesteps': candidate['timesteps'], 'physics_score': score,
                'surrogate_score': candidate['score'], 'checkpoint': str(path)})
        score, step, path = max(scores, key=lambda x: (x[0], -x[1]))
        choice = {'seed': seed, 'checkpoint': str(path), 'checkpoint_hash': file_hash(path),
                  'validation_score': score, 'timesteps': step}
        selected[algorithm].append(choice)
        atomic_json(out_dir/'physics_selected.json', choice)
physical_validation_df = pd.DataFrame(physical_validation_rows)
physical_validation_df.to_csv(RESULTS_DIR/'physics_validation.csv', index=False)
atomic_json(RESULTS_DIR/'selected_checkpoints.json', selected)
display(physical_validation_df)


## 13. Full testing under fixed conditions

Use six scheduling conditions: familiar P1–P4, added P5, added P6, doubled arrival rate, high drop cost,
and a drop cost outside the training range. Two lighting and color shifts bring the full-mode total to eight conditions.
Full mode runs 100 shifts per condition, policy, and training seed, saving and resuming at the shift level.
Quick mode checks the path with two shifts in each of three conditions: familiar, out-of-range drop cost, and color.

Compare surrogate, physics_camera, and physics_oracle using the same order and scene seeds.
Appearance conditions run only on the two physics backends that render images. Physical failures themselves
are not forced to match across policies. The oracle controls for localization; Clairvoyant CP-SAT is a separate
event-bound calculation with access to future orders.


In [ ]:
@dataclass(frozen=True)
class Condition:
    name: str
    products: tuple
    arrival_rate: float
    weights: tuple
    appearance: str = 'nominal'

ALL_CONDITIONS = [
    Condition('familiar', FAMILIAR, ENV.arrival_rate, EVAL_WEIGHTS),
    Condition('unseen_p5', FAMILIAR+('P5',), ENV.arrival_rate, EVAL_WEIGHTS),
    Condition('unseen_p6', FAMILIAR+('P6',), ENV.arrival_rate, EVAL_WEIGHTS),
    Condition('busy', FAMILIAR, 2*ENV.arrival_rate, EVAL_WEIGHTS),
    Condition('high_damage', FAMILIAR, ENV.arrival_rate, EVAL_WEIGHTS[:2]+(ENV.weight_high[2],)),
    Condition('damage_out_of_range', FAMILIAR, ENV.arrival_rate, EVAL_WEIGHTS[:2]+(2*ENV.weight_high[2],)),
    Condition('appearance_dim', FAMILIAR, ENV.arrival_rate, EVAL_WEIGHTS, 'dim'),
    Condition('appearance_color', FAMILIAR, ENV.arrival_rate, EVAL_WEIGHTS, 'color'),
]
conditions_by_name = {condition.name: condition for condition in ALL_CONDITIONS}
CONDITIONS = [conditions_by_name[name] for name in TEST_CONDITION_NAMES]
test_policies = {name: [(None, policy, {'rule': name, 'atcsr_parameters': ATCSR_PARAMETERS, 'rolling_cp_seconds': ROLLING_CP_SECONDS})]
    for name, policy in rule_baselines().items()}
for algorithm in ('ppo', 'dqn'):
    test_policies[algorithm.upper()] = [(choice['seed'], load_algorithm(algorithm, choice['checkpoint']),
        {'algorithm': algorithm, 'seed': choice['seed'], 'checkpoint_hash': choice['checkpoint_hash']})
        for choice in selected[algorithm]]

backends = {'surrogate': lambda: EventBackend(),
            'physics_camera': lambda: PhysicsBackend(WORKER),
            'physics_oracle': lambda: PhysicsBackend(WORKER, oracle=True)}
rows = []
for condition in CONDITIONS:
    cfg = replace(ENV, arrival_rate=condition.arrival_rate)
    shifts = make_instances(cfg, condition.products, N_TEST_SHIFTS, TEST_SEED, condition.appearance)
    for backend_name, backend_factory in backends.items():
        # Surrogate calibration is nominal; it cannot evaluate rendered appearance changes.
        if backend_name == 'surrogate' and condition.appearance != 'nominal':
            continue
        for policy_name, runs in test_policies.items():
            for seed, policy, identity in runs:
                signature = {'stage': 'test', 'backend': backend_name, 'policy': identity}
                directory = RESULTS_DIR/'trials'/condition.name/backend_name/policy_name/f'seed{seed}'
                results = cached_trials(policy, shifts, cfg, condition.weights, backend_factory, directory, signature)
                rows.extend({'condition': condition.name, 'backend': backend_name, 'policy': policy_name,
                             'run': -1 if seed is None else seed, **row} for row in results)
df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR/'test_shifts.csv', index=False)
display(df.head())


In [ ]:
ceiling_rows = []
for condition in CONDITIONS:
    if condition.appearance != 'nominal':
        continue
    cfg = replace(ENV, arrival_rate=condition.arrival_rate)
    shifts = make_instances(cfg, condition.products, N_TEST_SHIFTS, TEST_SEED, condition.appearance)
    directory = RESULTS_DIR/'ceilings'/condition.name
    directory.mkdir(parents=True, exist_ok=True)
    for shift, instance in enumerate(shifts):
        key = canonical_hash({'experiment': EXPERIMENT_KEY, 'cfg': asdict(cfg),
            'seconds': CLAIRVOYANT_CP_SECONDS, 'arrival': instance.arrival.tolist(),
            'deadline': instance.deadline.tolist(), 'weight': instance.weight.tolist(),
            'fragility': instance.fragility.tolist(), 'tray': instance.tray.tolist()})
        cache = directory/f'shift_{shift:03d}.json'
        if cache.exists():
            record = json.loads(cache.read_text())
            if record['key'] != key:
                raise RuntimeError('Clairvoyant cache differs; use a fresh RUN_DIR')
        else:
            record = {'key': key, **clairvoyant_ceiling(instance, cfg)}
            atomic_json(cache, record)
        ceiling_rows.append({'condition': condition.name, 'shift': shift, **record})
ceiling_df = pd.DataFrame(ceiling_rows)
ceiling_df.to_csv(RESULTS_DIR/'clairvoyant_reference.csv', index=False)
display(ceiling_df.groupby('condition')[['surrogate_score_upper_bound', 'on_time_upper_bound']].mean())


## 14. Means, uncertainty, and paired comparisons

Report hierarchical bootstrap CIs that resample RL seeds and shifts separately.
Compare each RL algorithm with EDF_modes, ATCSR+C, and Rolling CP-SAT on the same shifts; save win and tie rates.
Wilcoxon first averages scores across seeds, then uses one difference per shift. Its p-value therefore describes
the shift comparison for a fixed set of trained models; the separate bootstrap CI accounts for seed variation.
Declare all algorithm×reference×condition×backend comparisons as one family and apply Holm correction.

Report the clairvoyant reference and headroom only for the calibrated event environment.
The upper bound comes from an optimistic relaxation that removes failure, travel, and perception costs
and pocket constraints. Do not interpret it as the maximum physical score or as evidence that RL predicted future arrivals.


In [ ]:
def bootstrap_mean(matrix, rng, repeats=2000):
    matrix = np.asarray(matrix, dtype=float)
    if matrix.ndim != 2 or not matrix.size or not np.isfinite(matrix).all():
        raise ValueError('Bootstrap requires a complete run-by-shift matrix')
    values = []
    n_runs, n_shifts = matrix.shape
    for _ in range(repeats):
        runs = rng.integers(n_runs, size=n_runs)
        shifts = rng.integers(n_shifts, size=n_shifts)
        values.append(float(matrix[np.ix_(runs, shifts)].mean()))
    low, high = np.quantile(values, [.025, .975])
    return float(matrix.mean()), float(low), float(high)


def holm_adjust(p_values):
    values = np.asarray(p_values, dtype=float)
    if values.ndim != 1 or not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():
        raise ValueError('Invalid p-values')
    order = np.argsort(values, kind='stable')
    adjusted = np.empty_like(values)
    maximum = 0.
    for rank, index in enumerate(order):
        maximum = max(maximum, (len(values)-rank)*values[index])
        adjusted[index] = min(1., maximum)
    return adjusted


def paired_statistics(frame, references=('EDF_modes', 'ATCSR+C', 'Rolling CP-SAT'), repeats=2000):
    from scipy.stats import wilcoxon
    rng = np.random.default_rng(77_777)
    rows = []
    for (backend, condition), group in frame.groupby(['backend', 'condition'], sort=True):
        for reference in references:
            baseline = group[group['policy'].eq(reference)].set_index('shift')['score'].sort_index()
            if baseline.empty or baseline.index.has_duplicates:
                raise ValueError(f'Missing/duplicate reference shifts: {reference}')
            for policy in ('PPO', 'DQN'):
                matrix = group[group['policy'].eq(policy)].pivot(index='run', columns='shift', values='score')
                if matrix.empty or set(matrix.columns) != set(baseline.index):
                    raise ValueError(f'Unpaired shifts: {policy} vs {reference}')
                differences = matrix.to_numpy()-baseline.reindex(matrix.columns).to_numpy()[None, :]
                mean, low, high = bootstrap_mean(differences, rng, repeats)
                # One observation per shift, averaging trained seeds first. Do not pool seed×shift as independent.
                shift_delta = differences.mean(axis=0)
                p = 1. if np.all(shift_delta == 0.) else float(wilcoxon(
                    shift_delta, zero_method='pratt', alternative='two-sided', method='auto').pvalue)
                rows.append({'backend': backend, 'condition': condition, 'policy': policy,
                    'reference': reference, 'score_delta': mean, 'ci_low': low, 'ci_high': high,
                    'win_rate': float(np.mean(shift_delta > 0.)),
                    'tie_rate': float(np.mean(shift_delta == 0.)), 'p_value': p,
                    'seeds': len(matrix), 'paired_shifts': len(shift_delta)})
    result = pd.DataFrame(rows)
    # One predeclared family covers every algorithm×reference×condition×backend comparison.
    result['p_holm'] = holm_adjust(result['p_value'].to_numpy())
    result['significant_005'] = result['p_holm'] < .05
    return result


summary_rows = []
rng = np.random.default_rng(77_777)
for keys, group in df.groupby(['backend', 'condition', 'policy'], sort=True):
    backend, condition, policy = keys
    for metric in METRICS:
        matrix = group.pivot(index='run', columns='shift', values=metric).to_numpy()
        mean, low, high = bootstrap_mean(matrix, rng)
        summary_rows.append({'backend': backend, 'condition': condition, 'policy': policy,
            'metric': metric, 'mean': mean, 'ci_low': low, 'ci_high': high})
summary = pd.DataFrame(summary_rows)
summary.to_csv(RESULTS_DIR/'summary.csv', index=False)
display(summary.query("backend == 'physics_camera' and metric == 'score'").pivot(
    index='policy', columns='condition', values='mean').round(2))

paired = paired_statistics(df)
paired.to_csv(RESULTS_DIR/'paired_comparisons.csv', index=False)
display(paired.query("backend == 'physics_camera'").round(2))

camera = df[df['backend'].eq('physics_camera')].set_index(['condition', 'policy', 'run', 'shift'])
oracle = df[df['backend'].eq('physics_oracle')].set_index(['condition', 'policy', 'run', 'shift'])
perception_gap = camera[METRICS]-oracle[METRICS]
perception_gap.reset_index().to_csv(RESULTS_DIR/'camera_minus_oracle.csv', index=False)
display(perception_gap.groupby(['condition', 'policy'])[['score', 'failures', 'on_time']].mean().round(2))

# Only the calibrated event backend has the measured-duration CP-SAT upper bound.
headroom_rows = []
for condition, group in df[df['backend'].eq('surrogate')].groupby('condition'):
    reference = group[group['policy'].eq('ATCSR+C')].set_index('shift')['score']
    ceiling = ceiling_df[ceiling_df['condition'].eq(condition)].set_index('shift')['surrogate_score_upper_bound']
    for policy in ('PPO', 'DQN'):
        for run, values in group[group['policy'].eq(policy)].groupby('run'):
            scores = values.set_index('shift')['score'].sort_index()
            base = reference.reindex(scores.index)
            upper = ceiling.reindex(scores.index)
            if base.isna().any() or upper.isna().any():
                raise ValueError('Headroom references are not paired')
            gain = float((scores-base).mean())
            available = float((upper-base).mean())
            if np.any(scores.to_numpy() > upper.to_numpy()+1e-6):
                raise RuntimeError('A surrogate score exceeds the claimed relaxation bound')
            headroom_rows.append({'condition': condition, 'policy': policy, 'run': run,
                'reference': 'ATCSR+C', 'gain': gain, 'available': available,
                'fraction': gain/available if available > 1e-9 else None,
                'scope': 'calibrated_event_relaxation'})
headroom = pd.DataFrame(headroom_rows)
headroom.to_csv(RESULTS_DIR/'surrogate_headroom.csv', index=False)


In [ ]:
chart = summary.query("backend == 'physics_camera' and condition == 'familiar' and metric == 'score'").sort_values('mean')
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(chart['mean'], np.arange(len(chart)),
    xerr=np.vstack([chart['mean']-chart['ci_low'], chart['ci_high']-chart['mean']]), fmt='o', capsize=3)
ax.set_yticks(np.arange(len(chart)), chart['policy'])
ax.set(xlabel='Score, mean and 95% bootstrap interval', title='Familiar shifts: RGB + physics')
ax.grid(axis='x', alpha=.3)
fig.tight_layout()
fig.savefig(RESULTS_DIR/'familiar_scores.png', dpi=160)
plt.show()

result_manifest = {'mode': MODE, 'experiment_key': EXPERIMENT_KEY,
    'experiment_version': EXPERIMENT_VERSION, 'source': SOURCE_SPEC, 'assets': ASSET_MANIFEST,
    'calibration_key': CALIBRATION_KEY, 'calibration_data_hash': file_hash(CALIBRATION_PATH),
    'observation_data_hash': file_hash(OBSERVATION_PATH), 'camera_reference': CAMERA_REFERENCE,
    'selected': selected, 'seeds': SEEDS, 'val_seed': VAL_SEED, 'test_seed': TEST_SEED,
    'atcsr_parameters': ATCSR_PARAMETERS, 'observation_schema': OBS_SCHEMA,
    'solver_limits': {'rolling': ROLLING_CP_SECONDS, 'clairvoyant': CLAIRVOYANT_CP_SECONDS},
    'primary_baselines': [name for name in rule_baselines() if not name.endswith('_normal')],
    'ceiling_scope': 'calibrated_event_relaxation; not a physics ceiling',
    'validation_shifts': N_VAL_SHIFTS, 'test_shifts': N_TEST_SHIFTS,
    'conditions': [asdict(c) for c in CONDITIONS],
    'packages': {name: importlib.metadata.version(name) for name in
        ('numpy', 'torch', 'gymnasium', 'stable-baselines3', 'sb3-contrib', 'opencv-python-headless', 'imageio-ffmpeg', 'ortools', 'scipy')},
    'platform': platform.platform(), 'python': sys.version, 'simulator_runtime': WORKER.startup['runtime'],
    'camera_hardware_verified': {'megapixels': 2, 'fps': 30, 'interface': 'USB', 'cable_m': 3},
    'camera_simulation_assumptions': asdict(CELL),
    'limitations': ['fixed-height colored blocks', 'fragility is order metadata, not simulated fracture',
                    'IK executor, no ACT training', 'camera optical transform is a simulation assumption',
                    'camera/bracket payload and geometry are not modeled',
                    'no real-robot deployment or measured hand-eye calibration']}
atomic_json(RESULTS_DIR/'manifest.json', result_manifest)
score_table = summary[summary['metric'].eq('score')].copy()
# Use simple Markdown without an extra tabulate dependency.
report_lines = ['# RGB workcell experiment', '', f'Mode: {MODE}',
    f'Experiment version: {EXPERIMENT_VERSION}',
    f'Test shifts per condition/run: {N_TEST_SHIFTS}', f'Experiment key: {EXPERIMENT_KEY}', '',
    '| Backend | Condition | Policy | Score | 95% CI |', '|---|---|---|---:|---|']
for row in score_table.itertuples():
    report_lines.append(f'| {row.backend} | {row.condition} | {row.policy} | {row.mean:.3f} | [{row.ci_low:.3f}, {row.ci_high:.3f}] |')
report_lines += ['', 'Primary baselines: six mode-enabled rules, ATCSR+C, Rolling CP-SAT.',
    'Six normal-only rules are handling-mode ablations.',
    f'Observation profile: {OBSERVATION_PROFILE}, {OBS_SCHEMA["total_dim"]} values.',
    'Rolling CP-SAT optimality applies to its deterministic visible-queue approximation.',
    'Clairvoyant CP-SAT bounds the calibrated event relaxation; no physics headroom is claimed.',
    'See paired_comparisons.csv for paired CI, win rates, Wilcoxon and global Holm correction.',
    'Wilcoxon uses one mean-across-seeds difference per shift; CI also resamples training seeds.',
    'See surrogate_headroom.csv and clairvoyant_reference.csv for bound-based event headroom.',
    'See manifest.json for settings and limitations; test_shifts.csv for individual shifts.']
report_lines.extend(REPORT_MODE_NOTES)
(RESULTS_DIR/'report.md').write_text('\n'.join(report_lines)+'\n', encoding='utf-8')
print(f'Results saved in {RESULTS_DIR.resolve()}')


## 15. Pick videos from trained policies

Run the selected PPO and DQN checkpoints on the same familiar orders and scene.
Defaults are training seed 0, a 60-second shift, and 15 FPS. `quick` uses 30 seconds; adjust the duration with
`DEMO_SHIFT_SECONDS`. This cell loads existing checkpoints. Add `EDF_modes` to `DEMO_POLICIES` to compare a rule policy.

The video shows a workcell overview on the left and the policy's wrist RGB on the right, with pocket, normal/careful mode,
and time overlays. The additional overview is for recording; the policy continues to use wrist RGB detections
and order specifications. The worker records frames continuously during observation, approach, picking,
output verification, and return motion.

Save H.264 MP4 files, order/settings/checkpoint information, and action/observation events in `results/videos/`.
Metrics from these short runs are saved as visualization data. The preceding fixed-test results are in `test_shifts.csv`.
The final cell plays the videos. For saved videos, rerun only the playback cell.


In [ ]:
class RecordingPhysicsBackend(PhysicsBackend):
    def __init__(self, worker, path, fps, label, eye, target):
        super().__init__(worker)
        self.path, self.fps, self.label = Path(path), fps, label
        self.eye, self.target, self.recording = eye, target, False

    def reset(self, instance, arm=0):
        super().reset(instance, arm)
        self.worker.call('start_recording', path=str(self.path.resolve()), fps=self.fps,
                         label=self.label, eye=self.eye, target=self.target)
        self.recording = True

    def stop(self):
        result = self.worker.call('stop_recording')
        self.recording = False
        return result

    def abort(self):
        if self.recording:
            self.worker.call('abort_recording')
            self.recording = False


def record_policy_video(worker, policy, instance, cfg, weights, path, label,
                        fps=15, eye=(.55, -.60, .45), target=(.10, 0., .08)):
    backend = RecordingPhysicsBackend(worker, path, fps, label, eye, target)
    env = LVHMSchedulingEnv(cfg, backend)
    try:
        metrics = run_episode(env, policy, instance, weights)
        video = backend.stop()
        return {'label': label, 'video': video, 'metrics': metrics, 'events': env._events}
    finally:
        try:
            backend.abort()
        finally:
            env.close()


def selected_demo_checkpoint(choices, algorithm, seed):
    matches = [choice for choice in choices[algorithm] if choice['seed'] == seed]
    if len(matches) != 1:
        raise ValueError(f'Expected one selected {algorithm} checkpoint for seed {seed}')
    choice = matches[0]
    path = Path(choice['checkpoint'])
    if file_hash(path) != choice['checkpoint_hash']:
        raise RuntimeError(f'Selected checkpoint changed: {path}')
    return choice


In [ ]:
DEMO_POLICIES = ('PPO', 'DQN')
DEMO_TRAINING_SEED = SEEDS[0]
DEMO_SCENE_SEED = 50_000
DEMO_SHIFT_SECONDS = min(60., ENV.shift_length)
DEMO_FPS = 15
DEMO_EYE = (.55, -.60, .45)
DEMO_TARGET = (.10, 0., .08)
VIDEO_DIR = RESULTS_DIR / 'videos'
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

demo_cfg = replace(ENV, shift_length=DEMO_SHIFT_SECONDS)
demo_instance = make_instances(demo_cfg, FAMILIAR, 1, DEMO_SCENE_SEED)[0]
demo_choices = json.loads((RESULTS_DIR/'selected_checkpoints.json').read_text())
demo_rows = []
atomic_json(VIDEO_DIR/'index.json', demo_rows)
for name in DEMO_POLICIES:
    if name in ('PPO', 'DQN'):
        algorithm = name.lower()
        choice = selected_demo_checkpoint(demo_choices, algorithm, DEMO_TRAINING_SEED)
        policy = load_algorithm(algorithm, choice['checkpoint'])
        identity = {'algorithm': algorithm, 'seed': DEMO_TRAINING_SEED,
                    'checkpoint': choice['checkpoint'], 'checkpoint_hash': choice['checkpoint_hash']}
        label = f'{name} seed {DEMO_TRAINING_SEED}'
        stem = f'{algorithm}_seed{DEMO_TRAINING_SEED}'
    else:
        policy = rule_baselines()[name]
        identity, label, stem = {'rule': name}, name, name.lower()
    print(f'Recording {label}: {DEMO_SHIFT_SECONDS:g}s, {DEMO_FPS} FPS')
    result = record_policy_video(WORKER, policy, demo_instance, demo_cfg, EVAL_WEIGHTS,
        VIDEO_DIR/f'{stem}.mp4', label, DEMO_FPS, DEMO_EYE, DEMO_TARGET)
    events = result.pop('events')
    atomic_json(VIDEO_DIR/f'{stem}_events.json', events)
    result.update(policy=identity, experiment_version=EXPERIMENT_VERSION, experiment_key=EXPERIMENT_KEY,
        purpose='visualization', cfg=asdict(demo_cfg), weights=EVAL_WEIGHTS,
        order_seed=DEMO_SCENE_SEED, scene_seed=demo_instance.scene_seed,
        overview_eye=DEMO_EYE, overview_target=DEMO_TARGET,
        orders={'arrival': demo_instance.arrival.tolist(), 'deadline': demo_instance.deadline.tolist(),
                'weight': demo_instance.weight.tolist(), 'fragility': demo_instance.fragility.tolist(),
                'tray': demo_instance.tray.tolist()}, events_path=str(VIDEO_DIR/f'{stem}_events.json'))
    atomic_json(VIDEO_DIR/f'{stem}.json', result)
    demo_rows.append(result)
    atomic_json(VIDEO_DIR/'index.json', demo_rows)
display(pd.DataFrame([{'policy': row['label'], **row['metrics'],
                      'video_seconds': row['video']['duration_seconds']} for row in demo_rows]))


In [ ]:
from IPython.display import Markdown, Video, display

video_index = json.loads((RESULTS_DIR/'videos'/'index.json').read_text())
for entry in video_index:
    display(Markdown(f"**{entry['label']}** — Workcell overview / wrist RGB"))
    display(Video(filename=entry['video']['path'], embed=True, mimetype='video/mp4', width=960,
                  html_attributes='controls preload="metadata"'))


## 16. Shutdown and interpretation

Inspect the performance change from surrogate to physics and the gap from camera to oracle separately.
P5/P6 changes assess feature generalization; dim/color changes assess perception generalization.
Do not claim a benefit from careful mode unless a reduction in drops is measured.
If you run the optional 95-dimensional profile, compare it against the 79-dimensional default before claiming a benefit.
Beating Rolling CP-SAT alone does not establish an advantage from predicting future arrivals.
This experiment studies scheduling in an LVHM workcell that supplies blocks of fixed height one pocket at a time.
Transfer to hardware requires measurements of object geometry, camera calibration, grasp orientation,
contact coefficients, and servo limits.

Before hardware transfer, measure actual intrinsics, distortion, hand-eye calibration, field of view,
cable behavior, and mounted mass with the camera on the wrist.


In [ ]:
WORKER.close()


## References

- [OR-Tools CP-SAT](https://developers.google.com/optimization/cp/cp_solver): integer model, solver status and bounds.
- [Original ATCSR paper](https://doi.org/10.1016/j.cie.2007.08.011): setups and ready times dispatch rule.
- [SciPy Wilcoxon](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.wilcoxon.html): paired signed-rank test.

- [LeHome](https://github.com/lehome-official/lehome): reference for SO101 assets and Isaac camera/rigid-body workcells.
- [LeHome Challenge](https://github.com/lehome-official/lehome-challenge): reference evaluation structure. This notebook defines its own task and reward.
- [Pinned LeHome release assets](https://huggingface.co/datasets/lehome/lehome_release/tree/bce78287bf9510449fbe4a045ebb6d53ad4f3ed0/robots): SO101 USD and URDF.
- [WOWROBO SO-ARM101 kit](https://shop.wowrobo.com/products/so-arm101-diy-kit-assembled-version-1): package contents, including the camera.
- [WOWROBO 2MP USB camera](https://shop.wowrobo.com/products/2mp-usb-camera-module-for-so-arm100-101-30fps-3m-cable): listed 2MP, 30 FPS, USB, and 3m cable specifications.
- [Isaac Sim 6.0 installation](https://docs.isaacsim.omniverse.nvidia.com/6.0.0/installation/install_python.html),
  [system requirements](https://docs.isaacsim.omniverse.nvidia.com/6.0.0/installation/requirements.html).
- [Isaac Sim articulation controller](https://docs.isaacsim.omniverse.nvidia.com/6.0.0/robot_simulation/articulation_controller.html): radian commands and joint-index API.
- [Isaac Sim 6.0 camera migration guide](https://docs.isaacsim.omniverse.nvidia.com/6.0.0/migration_guides/isaac_sim_6_0/sensors_camera_to_experimental_rtx.html): RtxCamera, CameraSensor, RGB Warp arrays, and resolution ordering.
- [Isaac Sim RTX Camera API](https://docs.isaacsim.omniverse.nvidia.com/6.0.0/py/source/extensions/isaacsim.sensors.experimental.rtx/docs/index.html): RGB sensor configuration and data capture.

- [SO101 wrist camera mount](https://github.com/TheRobotStudio/SO-ARM100/blob/5f6d2b876a53a4872e405b991dd925556c9e38a4/Optional/SO101_Wrist_Cam_Hex-Nut_Mount_32x32_UVC_Module/README.md): mounting on the fixed gripper body.
- [Public SO101 camera URDF](https://github.com/TheRobotStudio/SO-ARM100/blob/5f6d2b876a53a4872e405b991dd925556c9e38a4/Simulation/SO101/so101_new_calib_camera.urdf): wrist-camera fixed-joint structure and bracket reference point.
- [OpenCV hand-eye calibration](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html): gripper FK, solvePnP target poses, camera-to-gripper estimation, and nonparallel rotation requirements.
- [Isaac Sim robot camera attachment](https://docs.isaacsim.omniverse.nvidia.com/6.0.0/robot_setup_tutorials/tutorial_gui_camera_sensors.html): configuring a camera beneath a robot link.
- [RTX camera API](https://docs.isaacsim.omniverse.nvidia.com/6.0.0/py/source/extensions/isaacsim.sensors.experimental.rtx/docs/index.html)
- [imageio-ffmpeg writer](https://github.com/imageio/imageio-ffmpeg/blob/v0.6.0/imageio_ffmpeg/_io.py)
